# Whisper H1-H5 settings, inference pilots, batch processing, and ASR QC

These are original, unmodified Python source cells from the exported Kaggle notebook, organized by research phase. Execution outputs and counts have been cleared. Cells are in original index order within this phase. The Kaggle project workspace and some earlier phases are required; this notebook is not guaranteed to execute by itself from a clean environment. The original index for every cell is stored in cell metadata.

The historical record includes exploratory retries and some superseded checks. Do not rerun an early output-producing cell over frozen results without backing up the project first.

In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

OUT = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT.parent.mkdir(parents=True, exist_ok=True)

config = {
    "configuration_name": "asr_hypothesis_pilot_v1",
    "status": "proposed_not_frozen",
    "dataset": "mispeech/speechocean762",
    "dataset_revision": (
        "06385584fad212b26134c656fdd3ccf9f093f33e"
    ),
    "split": "train",
    "model_id": "openai/whisper-small.en",
    "language": "english",
    "task": "transcribe",
    "pilot_audio_count": 10,
    "hypotheses": [
        {
            "id": "H1",
            "method": "greedy",
            "num_beams": 1,
            "do_sample": False
        },
        {
            "id": "H2",
            "method": "beam_search",
            "num_beams": 5,
            "do_sample": False
        },
        {
            "id": "H3",
            "method": "sampling",
            "do_sample": True,
            "temperature": 0.3,
            "seed": 101
        },
        {
            "id": "H4",
            "method": "sampling",
            "do_sample": True,
            "temperature": 0.3,
            "seed": 202
        },
        {
            "id": "H5",
            "method": "sampling",
            "do_sample": True,
            "temperature": 0.3,
            "seed": 303
        }
    ],
    "methodological_rules": {
        "use_prompt_text_as_asr_context": False,
        "use_human_fluency_for_asr_selection": False,
        "use_official_test_split": False,
        "preserve_raw_hypotheses": True,
        "evaluate_variability_before_full_run": True
    },
    "created_at": datetime.now(timezone.utc).isoformat()
}

if OUT.exists():
    previous = json.loads(OUT.read_text(encoding="utf-8"))

    def experiment_definition(x):
        return {
            k: v for k, v in x.items()
            if k != "created_at"
        }

    assert experiment_definition(previous) == experiment_definition(config), (
        "Existing ASR config differs. "
        "Do not overwrite without reviewing the changes."
    )

    print("Existing identical configuration found.")
else:
    OUT.write_text(
        json.dumps(config, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_asr_config_status"] = "proposed_not_frozen"
progress["step3_asr_pilot_config"] = str(OUT)
progress["next_task"] = (
    "Validate Whisper-small.en model compatibility and decoding support"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 3.2 ASR DESIGN SAVED ===")
print("Model:", config["model_id"])
print("Pilot recordings:", config["pilot_audio_count"])
print("Hypotheses:", len(config["hypotheses"]))

for h in config["hypotheses"]:
    print(h["id"], "-", h["method"])

print("Config status:", config["status"])
print("Config file:", OUT)
print("Next task:", progress["next_task"])
print("Checkpoint updated.")


In [ ]:

import json
import inspect
from pathlib import Path

import torch
import transformers

from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    GenerationConfig
)

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"

config = json.loads(CONFIG.read_text(encoding="utf-8"))

print("=== STEP 3.2: WHISPER API COMPATIBILITY ===")

print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("Model:", config["model_id"])
print("Configuration status:", config["status"])

print("\n=== MODEL CLASS ===")
print("Class:", WhisperForConditionalGeneration.__name__)
print("Processor:", WhisperProcessor.__name__)

signature = inspect.signature(
    WhisperForConditionalGeneration.generate
)

params = signature.parameters

print("\n=== GENERATE SIGNATURE ===")
print(signature)

print("\n=== GENERATION PARAMETERS ===")

for name in [
    "generation_config",
    "logits_processor",
    "stopping_criteria",
    "return_dict_in_generate",
    "kwargs"
]:
    print(name, ":", name in params)

print("\n=== GENERATION CONFIG SUPPORT ===")

gc = GenerationConfig()

for name, value in [
    ("do_sample", True),
    ("temperature", 0.3),
    ("num_beams", 5),
    ("max_new_tokens", 128)
]:
    setattr(gc, name, value)
    print(name, "=", getattr(gc, name))

print("\n=== EXPERIMENT VALIDATION ===")

hypotheses = config["hypotheses"]

assert len(hypotheses) == 5
assert len({h["id"] for h in hypotheses}) == 5

assert hypotheses[0]["method"] == "greedy"
assert hypotheses[1]["method"] == "beam_search"

assert all(
    h["method"] == "sampling"
    for h in hypotheses[2:]
)

seeds = [h["seed"] for h in hypotheses[2:]]
assert len(set(seeds)) == 3

print("Hypotheses:", len(hypotheses))
print("Sampling seeds:", seeds)
print("Static config validation: PASS")

print("\nNOTE:")
print("This check verifies API structure only.")
print("Actual generation behavior is not yet verified.")

print("\n=== API COMPATIBILITY CHECK COMPLETED ===")


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

from huggingface_hub import HfApi
from transformers import AutoConfig, AutoProcessor

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

CONFIG_FILE = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
REPORT = PROJECT / "reports/step03_whisper_model_revision.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

config = json.loads(CONFIG_FILE.read_text(encoding="utf-8"))
assert config["status"] == "proposed_not_frozen"

model_id = config["model_id"]

print("=== STEP 3.2: WHISPER MODEL REVISION CHECK ===")

api = HfApi()
model_info = api.model_info(repo_id=model_id)

revision = model_info.sha

assert revision is not None
assert len(revision) == 40

print("Model:", model_id)
print("Exact revision:", revision)

model_config = AutoConfig.from_pretrained(
    model_id,
    revision=revision
)

processor = AutoProcessor.from_pretrained(
    model_id,
    revision=revision
)

print("\n=== MODEL METADATA ===")
print("Model type:", model_config.model_type)
print("Encoder layers:", model_config.encoder_layers)
print("Decoder layers:", model_config.decoder_layers)
print("Max source positions:", model_config.max_source_positions)
print("Feature extractor sample rate:",
      processor.feature_extractor.sampling_rate)

assert model_config.model_type == "whisper"
assert processor.feature_extractor.sampling_rate == 16000

# Do not silently replace an existing pinned revision.
if "model_revision" in config:
    assert config["model_revision"] == revision, (
        "Previously pinned model revision differs. "
        "Review before proceeding."
    )
else:
    config["model_revision"] = revision

CONFIG_FILE.write_text(
    json.dumps(config, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

report = {
    "model_id": model_id,
    "model_revision": revision,
    "model_type": model_config.model_type,
    "encoder_layers": model_config.encoder_layers,
    "decoder_layers": model_config.decoder_layers,
    "sample_rate": processor.feature_extractor.sampling_rate,
    "transformers_version": "5.16.1",
    "status": "PASS",
    "checked_at": datetime.now(timezone.utc).isoformat()
}

REPORT.parent.mkdir(parents=True, exist_ok=True)
REPORT.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_whisper_revision"] = revision
progress["step3_model_metadata_validation"] = "PASS"
progress["next_task"] = (
    "Load pinned Whisper model and run a small inference test"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== MODEL REVISION SAVED ===")
print("Revision:", revision)
print("Config status:", config["status"])
print("Report:", REPORT)
print("Next task:", progress["next_task"])
print("Checkpoint updated.")


In [ ]:

import io
import json
import time
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import torch
import transformers

from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor
)

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
OUT = PROJECT / "features/asr/asr_smoke_test_096140003_H1.csv"
REPORT = PROJECT / "reports/step03_whisper_smoke_test.json"

config = json.loads(CONFIG.read_text(encoding="utf-8"))
MODEL_ID = config["model_id"]
REVISION = config["model_revision"]
TARGET_ID = "096140003"

OUT.parent.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "GPU is required for this test"

device = "cuda:0"

print("=== STEP 3.2: WHISPER GREEDY SMOKE TEST ===")
print("Model:", MODEL_ID)
print("Revision:", REVISION)
print("Device:", device)

# Locate exactly one audio from train split
pf = pq.ParquetFile(TRAIN)
matches = []

for group_id in range(pf.metadata.num_row_groups):
    audios = pf.read_row_group(
        group_id, columns=["audio"]
    ).column("audio").to_pylist()

    for item in audios:
        if Path(item["path"]).stem == TARGET_ID:
            matches.append(item["bytes"])

assert len(matches) == 1, (
    f"Expected exactly one audio, found {len(matches)}"
)

y, sr = sf.read(io.BytesIO(matches[0]), dtype="float32")

assert sr == 16000 and y.ndim == 1
assert np.isfinite(y).all()

print("Audio ID:", TARGET_ID)
print("Duration:", round(len(y) / sr, 3), "seconds")

# Load the pinned processor and model
processor = WhisperProcessor.from_pretrained(
    MODEL_ID,
    revision=REVISION
)

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
    revision=REVISION,
    torch_dtype=torch.float16
)

model = model.to(device)
model.eval()

inputs = processor(
    y,
    sampling_rate=sr,
    return_tensors="pt"
)

input_features = inputs.input_features.to(
    device=device,
    dtype=torch.float16
)

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

start = time.perf_counter()

with torch.inference_mode():
    generated_ids = model.generate(
        input_features,
        do_sample=False,
        num_beams=1,
        max_new_tokens=128,
        task="transcribe"
    )

if torch.cuda.is_available():
    torch.cuda.synchronize()

elapsed = time.perf_counter() - start

transcription = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True
)[0].strip()

result = pd.DataFrame([{
    "audio_id": TARGET_ID,
    "hypothesis_id": "H1",
    "method": "greedy",
    "model_id": MODEL_ID,
    "model_revision": REVISION,
    "transcription": transcription,
    "duration_sec": len(y) / sr,
    "inference_sec": elapsed
}])

result.to_csv(OUT, index=False, encoding="utf-8")

report = {
    "audio_id": TARGET_ID,
    "hypothesis_id": "H1",
    "model_id": MODEL_ID,
    "model_revision": REVISION,
    "transformers_version": transformers.__version__,
    "torch_version": torch.__version__,
    "device": torch.cuda.get_device_name(0),
    "dtype": "float16",
    "do_sample": False,
    "num_beams": 1,
    "max_new_tokens": 128,
    "generated_token_count": int(generated_ids.shape[-1]),
    "transcription": transcription,
    "inference_sec": elapsed,
    "created_at": datetime.now(timezone.utc).isoformat()
}

REPORT.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== H1 GREEDY RESULT ===")
print("Audio ID:", TARGET_ID)
print("Transcription:", repr(transcription))
print("Generated tokens:", generated_ids.shape[-1])
print("Inference time:", round(elapsed, 3), "seconds")
print("Output CSV:", OUT)
print("Report:", REPORT)
print("Status: COMPLETED")


In [ ]:

import io
import json
import time
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import torch
import transformers

from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor
)

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
OUT = PROJECT / "features/asr/asr_smoke_test_096140003_H1.csv"
REPORT = PROJECT / "reports/step03_whisper_smoke_test.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

config = json.loads(CONFIG.read_text(encoding="utf-8"))

MODEL_ID = config["model_id"]
REVISION = config["model_revision"]
TARGET_ID = "096140003"

OUT.parent.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

assert MODEL_ID == "openai/whisper-small.en"
assert torch.cuda.is_available()

device = "cuda:0"

print("=== STEP 3.2: CORRECTED GREEDY SMOKE TEST ===")
print("Model:", MODEL_ID)
print("Revision:", REVISION)

# Load target audio
pf = pq.ParquetFile(TRAIN)
matches = []

for group_id in range(pf.metadata.num_row_groups):
    audios = pf.read_row_group(
        group_id, columns=["audio"]
    ).column("audio").to_pylist()

    for item in audios:
        if Path(item["path"]).stem == TARGET_ID:
            matches.append(item["bytes"])

assert len(matches) == 1

y, sr = sf.read(
    io.BytesIO(matches[0]),
    dtype="float32"
)

assert sr == 16000
assert y.ndim == 1
assert np.isfinite(y).all()

print("Audio ID:", TARGET_ID)
print("Duration:", round(len(y) / sr, 3), "seconds")

# Load processor and model
processor = WhisperProcessor.from_pretrained(
    MODEL_ID,
    revision=REVISION
)

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
    revision=REVISION,
    dtype=torch.float16
).to(device)

model.eval()

inputs = processor(
    y,
    sampling_rate=sr,
    return_tensors="pt"
)

input_features = inputs.input_features.to(
    device=device,
    dtype=torch.float16
)

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

torch.cuda.synchronize()
start = time.perf_counter()

with torch.inference_mode():
    generated_ids = model.generate(
        input_features,
        do_sample=False,
        num_beams=1,
        max_new_tokens=128
        # No task or language for English-only Whisper
    )

torch.cuda.synchronize()
elapsed = time.perf_counter() - start

transcription = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True
)[0].strip()

assert isinstance(transcription, str)

result = pd.DataFrame([{
    "audio_id": TARGET_ID,
    "hypothesis_id": "H1",
    "method": "greedy",
    "model_id": MODEL_ID,
    "model_revision": REVISION,
    "transcription": transcription,
    "duration_sec": len(y) / sr,
    "inference_sec": elapsed
}])

result.to_csv(OUT, index=False, encoding="utf-8")

report = {
    "audio_id": TARGET_ID,
    "hypothesis_id": "H1",
    "model_id": MODEL_ID,
    "model_revision": REVISION,
    "transformers_version": transformers.__version__,
    "torch_version": torch.__version__,
    "device": torch.cuda.get_device_name(0),
    "dtype": "float16",
    "do_sample": False,
    "num_beams": 1,
    "max_new_tokens": 128,
    "task_argument": None,
    "language_argument": None,
    "generated_token_count": int(generated_ids.shape[-1]),
    "transcription": transcription,
    "inference_sec": elapsed,
    "created_at": datetime.now(timezone.utc).isoformat()
}

REPORT.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_h1_smoke_test"] = "completed"
progress["step3_h1_smoke_test_audio"] = TARGET_ID
progress["next_task"] = (
    "Evaluate H1 transcription and test beam search H2"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== H1 GREEDY RESULT ===")
print("Audio ID:", TARGET_ID)
print("Transcription:", repr(transcription))
print("Generated tokens:", generated_ids.shape[-1])
print("Inference time:", round(elapsed, 3), "seconds")
print("Output CSV:", OUT)
print("Report:", REPORT)
print("Checkpoint updated.")
print("Status: COMPLETED")


In [ ]:

import json
import time
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import torch

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
H1_FILE = PROJECT / "features/asr/asr_smoke_test_096140003_H1.csv"
H2_FILE = PROJECT / "features/asr/asr_smoke_test_096140003_H2.csv"
REPORT = PROJECT / "reports/step03_whisper_beam_test.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

config = json.loads(CONFIG.read_text(encoding="utf-8"))

assert "model" in globals(), "Whisper model not loaded"
assert "processor" in globals(), "Whisper processor not loaded"
assert "input_features" in globals(), "Audio features not loaded"

assert H1_FILE.exists()

h1 = pd.read_csv(H1_FILE, keep_default_na=False).iloc[0]
assert h1["audio_id"] == "096140003"
assert h1["hypothesis_id"] == "H1"

print("=== STEP 3.2: H2 BEAM SEARCH TEST ===")
print("Model:", config["model_id"])
print("Revision:", config["model_revision"])
print("Audio ID:", h1["audio_id"])

model.eval()
torch.cuda.synchronize()
start = time.perf_counter()

with torch.inference_mode():
    generated_ids_h2 = model.generate(
        input_features,
        do_sample=False,
        num_beams=5,
        max_new_tokens=128
    )

torch.cuda.synchronize()
elapsed = time.perf_counter() - start

transcription_h2 = processor.batch_decode(
    generated_ids_h2,
    skip_special_tokens=True
)[0].strip()

transcription_h1 = h1["transcription"]

h2 = pd.DataFrame([{
    "audio_id": "096140003",
    "hypothesis_id": "H2",
    "method": "beam_search",
    "num_beams": 5,
    "model_id": config["model_id"],
    "model_revision": config["model_revision"],
    "transcription": transcription_h2,
    "inference_sec": elapsed
}])

H2_FILE.parent.mkdir(parents=True, exist_ok=True)
H2_FILE.to_csv(H2_FILE, index=False)

report = {
    "audio_id": "096140003",
    "model_id": config["model_id"],
    "model_revision": config["model_revision"],
    "h1_transcription": transcription_h1,
    "h2_transcription": transcription_h2,
    "h1_h2_identical": transcription_h1 == transcription_h2,
    "h2_num_beams": 5,
    "h2_inference_sec": elapsed,
    "status": "completed",
    "created_at": datetime.now(timezone.utc).isoformat()
}

REPORT.parent.mkdir(parents=True, exist_ok=True)
REPORT.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_h2_smoke_test"] = "completed"
progress["next_task"] = (
    "Compare H1/H2 and test sampling H3-H5"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== H1 VS H2 COMPARISON ===")
print("H1 (greedy):", repr(transcription_h1))
print("H2 (beam):", repr(transcription_h2))
print("Identical:", report["h1_h2_identical"])
print("Beam inference time:", round(elapsed, 3), "seconds")
print("H2 CSV:", H2_FILE)
print("Checkpoint updated.")


In [ ]:

import json
import time
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import torch

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
H1_FILE = PROJECT / "features/asr/asr_smoke_test_096140003_H1.csv"
H2_FILE = PROJECT / "features/asr/asr_smoke_test_096140003_H2.csv"
REPORT = PROJECT / "reports/step03_whisper_beam_test.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

config = json.loads(CONFIG.read_text(encoding="utf-8"))

assert "model" in globals(), "Whisper model not loaded"
assert "processor" in globals(), "Whisper processor not loaded"
assert "input_features" in globals(), "Audio features not loaded"

assert H1_FILE.exists()

# IMPORTANT: Preserve leading zeros in audio IDs.
h1_df = pd.read_csv(
    H1_FILE,
    dtype={"audio_id": "string"},
    keep_default_na=False
)

assert len(h1_df) == 1

h1 = h1_df.iloc[0]

assert str(h1["audio_id"]) == "096140003"
assert h1["hypothesis_id"] == "H1"
assert h1["model_id"] == config["model_id"]
assert h1["model_revision"] == config["model_revision"]

print("=== STEP 3.2: H2 BEAM SEARCH TEST ===")
print("Model:", config["model_id"])
print("Revision:", config["model_revision"])
print("Audio ID:", h1["audio_id"])

# Same model and same audio; only decoding changes.
model.eval()

torch.cuda.synchronize()
start = time.perf_counter()

with torch.inference_mode():
    generated_ids_h2 = model.generate(
        input_features,
        do_sample=False,
        num_beams=5,
        max_new_tokens=128
    )

torch.cuda.synchronize()
elapsed = time.perf_counter() - start

transcription_h2 = processor.batch_decode(
    generated_ids_h2,
    skip_special_tokens=True
)[0].strip()

transcription_h1 = str(h1["transcription"])

h2_df = pd.DataFrame([{
    "audio_id": "096140003",
    "hypothesis_id": "H2",
    "method": "beam_search",
    "num_beams": 5,
    "model_id": config["model_id"],
    "model_revision": config["model_revision"],
    "transcription": transcription_h2,
    "inference_sec": elapsed
}])

H2_FILE.parent.mkdir(parents=True, exist_ok=True)
h2_df.to_csv(H2_FILE, index=False, encoding="utf-8")

report = {
    "audio_id": "096140003",
    "model_id": config["model_id"],
    "model_revision": config["model_revision"],
    "h1_transcription": transcription_h1,
    "h2_transcription": transcription_h2,
    "h1_h2_identical": transcription_h1 == transcription_h2,
    "h2_num_beams": 5,
    "h2_inference_sec": elapsed,
    "status": "completed",
    "created_at": datetime.now(timezone.utc).isoformat()
}

REPORT.parent.mkdir(parents=True, exist_ok=True)
REPORT.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_h2_smoke_test"] = "completed"
progress["next_task"] = "Compare H1/H2 and test sampling H3-H5"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== H1 VS H2 COMPARISON ===")
print("H1 (greedy):", repr(transcription_h1))
print("H2 (beam):", repr(transcription_h2))
print("Identical:", report["h1_h2_identical"])
print("Beam inference time:", round(elapsed, 3), "seconds")
print("H2 CSV:", H2_FILE)
print("Checkpoint updated.")


In [ ]:

import json
import time
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import torch
import transformers

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
OUT_DIR = PROJECT / "features/asr"
REPORT = PROJECT / "reports/step03_sampling_smoke_test.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

config = json.loads(CONFIG.read_text(encoding="utf-8"))

assert "model" in globals(), "Whisper model not loaded"
assert "processor" in globals(), "Whisper processor not loaded"
assert "input_features" in globals(), "Audio features not loaded"

assert config["status"] == "proposed_not_frozen"

TARGET_ID = "096140003"

OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

# Read saved deterministic hypotheses
transcriptions = {}

for hypothesis_id in ["H1", "H2"]:
    path = OUT_DIR / f"asr_smoke_test_{TARGET_ID}_{hypothesis_id}.csv"

    df = pd.read_csv(
        path,
        dtype={"audio_id": "string"},
        keep_default_na=False
    )

    assert len(df) == 1
    assert df.iloc[0]["audio_id"] == TARGET_ID
    assert df.iloc[0]["hypothesis_id"] == hypothesis_id

    transcriptions[hypothesis_id] = str(
        df.iloc[0]["transcription"]
    )

print("=== STEP 3.2: STOCHASTIC SAMPLING TEST ===")
print("Model:", config["model_id"])
print("Audio:", TARGET_ID)

sampling_hypotheses = [
    h for h in config["hypotheses"]
    if h["method"] == "sampling"
]

assert len(sampling_hypotheses) == 3

records = []

model.eval()

for h in sampling_hypotheses:

    hypothesis_id = h["id"]
    seed = int(h["seed"])
    temperature = float(h["temperature"])

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.inference_mode():
        generated_ids = model.generate(
            input_features,
            do_sample=True,
            num_beams=1,
            temperature=temperature,
            max_new_tokens=128
        )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    transcription = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0].strip()

    record = {
        "audio_id": TARGET_ID,
        "hypothesis_id": hypothesis_id,
        "method": "sampling",
        "model_id": config["model_id"],
        "model_revision": config["model_revision"],
        "do_sample": True,
        "num_beams": 1,
        "temperature": temperature,
        "seed": seed,
        "transcription": transcription,
        "generated_tokens": int(generated_ids.shape[-1]),
        "inference_sec": elapsed
    }

    records.append(record)
    transcriptions[hypothesis_id] = transcription

    output = OUT_DIR / f"asr_smoke_test_{TARGET_ID}_{hypothesis_id}.csv"

    pd.DataFrame([record]).to_csv(
        output,
        index=False,
        encoding="utf-8"
    )

    print(
        f"{hypothesis_id} | seed={seed} | "
        f"time={elapsed:.3f}s | {transcription!r}"
    )

unique_transcripts = len(set(transcriptions.values()))

report = {
    "audio_id": TARGET_ID,
    "model_id": config["model_id"],
    "model_revision": config["model_revision"],
    "hypotheses": transcriptions,
    "unique_transcriptions": unique_transcripts,
    "total_hypotheses": 5,
    "all_identical": unique_transcripts == 1,
    "status": "completed",
    "created_at": datetime.now(timezone.utc).isoformat()
}

REPORT.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_sampling_smoke_test"] = "completed"
progress["step3_smoke_unique_transcriptions"] = unique_transcripts
progress["next_task"] = (
    "Assess H1-H5 variability and finalize ASR pilot design"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== H1-H5 TRANSCRIPTION COMPARISON ===")

for key, value in transcriptions.items():
    print(f"{key}: {value!r}")

print("\nTotal hypotheses:", len(transcriptions))
print("Unique transcriptions:", unique_transcripts)
print("All identical:", report["all_identical"])
print("Report:", REPORT)
print("Checkpoint updated.")


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
OUT = PROJECT / "data/metadata/asr_pilot_selection_10.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

config = json.loads(CONFIG.read_text(encoding="utf-8"))

assert len(pilot) == 400
assert pilot["audio_id"].is_unique
assert config["pilot_audio_count"] == 10

SMOKE_ID = "096140003"
SELECTION_SEED = 42

smoke = pilot.loc[
    pilot["audio_id"] == SMOKE_ID
].copy()

assert len(smoke) == 1

smoke_speaker = smoke.iloc[0]["speaker"]

# One audio per speaker for broader coverage
remaining = pilot.loc[
    pilot["speaker"] != smoke_speaker
].copy()

speakers = sorted(remaining["speaker"].unique())

rng = np.random.default_rng(SELECTION_SEED)

selected_speakers = rng.choice(
    speakers,
    size=9,
    replace=False
)

selected_rows = []

for speaker in selected_speakers:
    options = remaining.loc[
        remaining["speaker"] == speaker
    ].sort_values("pilot_index")

    selected_position = int(
        rng.integers(0, len(options))
    )

    selected_rows.append(
        options.iloc[selected_position]
    )

additional = pd.DataFrame(selected_rows)

selected = pd.concat(
    [smoke, additional],
    ignore_index=True
)

selected = selected.sort_values(
    "pilot_index"
).reset_index(drop=True)

assert len(selected) == 10
assert selected["audio_id"].is_unique
assert selected["speaker"].nunique() == 10
assert SMOKE_ID in set(selected["audio_id"])

selected["asr_pilot_order"] = range(1, 11)
selected["selection_method"] = (
    "one_recording_per_speaker_seed42"
)

if OUT.exists():
    old = pd.read_csv(
        OUT,
        dtype={"audio_id": "string", "speaker": "string"}
    )

    assert old["audio_id"].tolist() == selected["audio_id"].tolist(), (
        "Existing pilot selection differs. "
        "Do not silently overwrite."
    )
else:
    selected.to_csv(OUT, index=False, encoding="utf-8")

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_asr_pilot_selection_count"] = 10
progress["step3_asr_pilot_selection_seed"] = SELECTION_SEED
progress["step3_asr_pilot_selection_file"] = str(OUT)
progress["next_task"] = "Generate five ASR hypotheses for 10 pilot audios"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 3.2 ASR PILOT SELECTION ===")
print("Selected recordings:", len(selected))
print("Unique speakers:", selected["speaker"].nunique())
print("Random seed:", SELECTION_SEED)

print("\n=== SELECTED AUDIO IDs ===")
print(
    selected[
        ["asr_pilot_order", "pilot_index", "audio_id", "speaker"]
    ].to_string(index=False)
)

print("\nSelection file:", OUT)
print("Checkpoint updated.")


In [ ]:

import io
import json
import time
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import torch

from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor
)

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
SELECTION = PROJECT / "data/metadata/asr_pilot_selection_10.csv"
CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT_DIR = PROJECT / "features/asr/pilot"
OUT_DIR.mkdir(parents=True, exist_ok=True)

config = json.loads(CONFIG.read_text(encoding="utf-8"))
assert config["status"] == "proposed_not_frozen"

selection = pd.read_csv(
    SELECTION,
    dtype={"audio_id": "string", "speaker": "string"}
).sort_values("asr_pilot_order")

assert len(selection) == 10
assert selection["audio_id"].is_unique
assert selection["speaker"].nunique() == 10

hypotheses = config["hypotheses"]
assert [h["id"] for h in hypotheses] == [
    "H1", "H2", "H3", "H4", "H5"
]

MODEL_ID = config["model_id"]
REVISION = config["model_revision"]
device = "cuda:0"

assert torch.cuda.is_available()

# Find exactly the selected recordings.
wanted = set(selection["audio_id"])
audio_data = {}

pf = pq.ParquetFile(TRAIN)

for group_id in range(pf.metadata.num_row_groups):
    table = pf.read_row_group(
        group_id, columns=["audio"]
    )

    for item in table.column("audio").to_pylist():
        audio_id = Path(item["path"]).stem

        if audio_id in wanted:
            assert audio_id not in audio_data
            audio_data[audio_id] = item["bytes"]

assert set(audio_data) == wanted

# Load pinned ASR model.
processor = WhisperProcessor.from_pretrained(
    MODEL_ID,
    revision=REVISION
)

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
    revision=REVISION,
    dtype=torch.float16
).to(device)

model.eval()

print("=== STEP 3.2: ASR PILOT GENERATION ===")
print("Model:", MODEL_ID)
print("Revision:", REVISION)
print("Selected audios:", len(selection))
print("Hypotheses per audio:", len(hypotheses))

for _, item in selection.iterrows():

    audio_id = str(item["audio_id"])
    speaker = str(item["speaker"])

    output = OUT_DIR / f"asr_pilot_{audio_id}_H1-H5.csv"

    if output.exists():
        previous = pd.read_csv(
            output,
            dtype={"audio_id": "string", "speaker": "string"},
            keep_default_na=False
        )

        assert len(previous) == 5
        assert previous["audio_id"].eq(audio_id).all()
        assert previous["hypothesis_id"].tolist() == [
            "H1", "H2", "H3", "H4", "H5"
        ]
        assert previous["model_revision"].eq(REVISION).all()
        assert previous["transcription"].map(
            lambda x: isinstance(x, str)
        ).all()

        print(f"SKIP completed: {audio_id}")
        continue

    y, sr = sf.read(
        io.BytesIO(audio_data[audio_id]),
        dtype="float32"
    )

    assert sr == 16000 and y.ndim == 1
    assert np.isfinite(y).all()

    inputs = processor(
        y,
        sampling_rate=sr,
        return_tensors="pt"
    )

    input_features = inputs.input_features.to(
        device=device,
        dtype=torch.float16
    )

    records = []

    for h in hypotheses:

        hid = h["id"]
        method = h["method"]

        kwargs = {
            "max_new_tokens": 128,
            "num_beams": 1,
            "do_sample": False
        }

        seed = None
        temperature = None

        if method == "beam_search":
            kwargs["num_beams"] = h["num_beams"]

        elif method == "sampling":
            seed = int(h["seed"])
            temperature = float(h["temperature"])

            kwargs["do_sample"] = True
            kwargs["temperature"] = temperature

            torch.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)

        torch.cuda.synchronize()
        start = time.perf_counter()

        with torch.inference_mode():
            generated_ids = model.generate(
                input_features,
                **kwargs
            )

        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start

        transcription = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )[0].strip()

        records.append({
            "audio_id": audio_id,
            "speaker": speaker,
            "hypothesis_id": hid,
            "method": method,
            "model_id": MODEL_ID,
            "model_revision": REVISION,
            "do_sample": kwargs["do_sample"],
            "num_beams": kwargs["num_beams"],
            "temperature": temperature,
            "seed": seed,
            "transcription": transcription,
            "duration_sec": len(y) / sr,
            "inference_sec": elapsed
        })

    result = pd.DataFrame(records)

    assert len(result) == 5
    assert result["hypothesis_id"].is_unique

    # Atomic save to protect against interrupted writes.
    temp = output.with_suffix(".tmp")
    result.to_csv(temp, index=False, encoding="utf-8")
    temp.replace(output)

    unique = result["transcription"].nunique()

    print(
        f"COMPLETE: {audio_id} | "
        f"unique transcripts: {unique}/5"
    )

# Verify all pilot files.
all_results = []

for audio_id in selection["audio_id"]:
    path = OUT_DIR / f"asr_pilot_{audio_id}_H1-H5.csv"

    df = pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"},
        keep_default_na=False
    )

    assert len(df) == 5
    assert df["audio_id"].eq(audio_id).all()
    assert df["hypothesis_id"].tolist() == [
        "H1", "H2", "H3", "H4", "H5"
    ]

    all_results.append(df)

combined = pd.concat(all_results, ignore_index=True)

assert len(combined) == 50

unique_per_audio = combined.groupby(
    "audio_id"
)["transcription"].nunique()

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_asr_pilot_generated_audios"] = 10
progress["step3_asr_pilot_generated_hypotheses"] = 50
progress["step3_asr_pilot_status"] = "generated_pending_qc"
progress["next_task"] = (
    "Analyze ASR transcription variability across 10 pilot audios"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== ASR PILOT SUMMARY ===")
print("Audio count:", len(unique_per_audio))
print("Total hypotheses:", len(combined))
print("Audios with >1 transcription:",
      int((unique_per_audio > 1).sum()))
print("Mean unique transcripts:",
      round(unique_per_audio.mean(), 3))

print("\n=== UNIQUE TRANSCRIPTS PER AUDIO ===")
print(unique_per_audio.to_string())

print("\nOutput folder:", OUT_DIR)
print("Checkpoint updated.")


In [ ]:

import json
import re
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

ASR_DIR = PROJECT / "features/asr/pilot"
SELECTION = PROJECT / "data/metadata/asr_pilot_selection_10.csv"
OUT = PROJECT / "reports/step03_pilot_variability_per_audio.csv"
SUMMARY = PROJECT / "reports/step03_pilot_variability_summary.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

selection = pd.read_csv(
    SELECTION,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(selection) == 10

def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s']", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, start=1):
        curr = [i] + [0] * len(b)
        for j, y in enumerate(b, start=1):
            curr[j] = min(
                prev[j] + 1,
                curr[j-1] + 1,
                prev[j-1] + (x != y)
            )
        prev = curr
    return prev[-1]

def pair_metrics(reference, hypothesis):
    ref_words = reference.split()
    hyp_words = hypothesis.split()

    wer = (
        edit_distance(ref_words, hyp_words) / len(ref_words)
        if ref_words else (0.0 if not hyp_words else np.nan)
    )

    ref_chars = list(reference.replace(" ", ""))
    hyp_chars = list(hypothesis.replace(" ", ""))

    cer = (
        edit_distance(ref_chars, hyp_chars) / len(ref_chars)
        if ref_chars else (0.0 if not hyp_chars else np.nan)
    )

    return wer, cer

rows = []
all_hypotheses = []

for _, selected in selection.iterrows():

    audio_id = selected["audio_id"]
    source = ASR_DIR / f"asr_pilot_{audio_id}_H1-H5.csv"

    df = pd.read_csv(
        source,
        dtype={"audio_id": "string", "speaker": "string"},
        keep_default_na=False
    )

    assert len(df) == 5
    assert df["audio_id"].eq(audio_id).all()
    assert df["hypothesis_id"].tolist() == [
        "H1", "H2", "H3", "H4", "H5"
    ]

    df["normalized"] = df["transcription"].map(normalize_text)
    all_hypotheses.append(df)

    h1 = df.iloc[0]["normalized"]

    metric_rows = [
        pair_metrics(h1, text)
        for text in df.iloc[1:]["normalized"]
    ]

    valid_wer = [x[0] for x in metric_rows if np.isfinite(x[0])]
    valid_cer = [x[1] for x in metric_rows if np.isfinite(x[1])]

    rows.append({
        "audio_id": audio_id,
        "speaker": selected["speaker"],
        "raw_unique": df["transcription"].nunique(),
        "lexical_unique": df["normalized"].nunique(),
        "mean_wer_vs_h1": (
            float(np.mean(valid_wer)) if valid_wer else np.nan
        ),
        "max_wer_vs_h1": (
            float(np.max(valid_wer)) if valid_wer else np.nan
        ),
        "mean_cer_vs_h1": (
            float(np.mean(valid_cer)) if valid_cer else np.nan
        ),
        "empty_transcriptions": int(
            df["normalized"].eq("").sum()
        )
    })

result = pd.DataFrame(rows)

assert len(result) == 10
assert result["audio_id"].is_unique

OUT.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(OUT, index=False, encoding="utf-8")

combined = pd.concat(all_hypotheses, ignore_index=True)

summary = {
    "total_audio": len(result),
    "total_hypotheses": len(combined),
    "raw_variable_audio_count": int(
        (result["raw_unique"] > 1).sum()
    ),
    "lexically_variable_audio_count": int(
        (result["lexical_unique"] > 1).sum()
    ),
    "mean_raw_unique": float(result["raw_unique"].mean()),
    "mean_lexical_unique": float(result["lexical_unique"].mean()),
    "mean_wer_vs_h1": float(result["mean_wer_vs_h1"].mean()),
    "mean_cer_vs_h1": float(result["mean_cer_vs_h1"].mean()),
    "empty_transcriptions": int(
        result["empty_transcriptions"].sum()
    ),
    "note": (
        "WER/CER compare ASR hypotheses against H1, "
        "not against a verified human transcription."
    ),
    "created_at": datetime.now(timezone.utc).isoformat()
}

SUMMARY.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_pilot_variability_analysis"] = "completed"
progress["next_task"] = (
    "Review lexical variability and determine full ASR design"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 3.2 PILOT VARIABILITY SUMMARY ===")
for key, value in summary.items():
    if key != "created_at":
        print(f"{key}: {value}")

print("\n=== PER AUDIO VARIABILITY ===")
print(result.round(4).to_string(index=False))

print("\nReport CSV:", OUT)
print("Summary JSON:", SUMMARY)
print("Checkpoint updated.")


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
ASR_DIR = PROJECT / "features/asr/pilot"
SELECTION = PROJECT / "data/metadata/asr_pilot_selection_10.csv"
OUT = PROJECT / "reports/step03_pilot_hypothesis_content_audit.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

selection = pd.read_csv(
    SELECTION,
    dtype={"audio_id": "string", "speaker": "string"}
)

records = []

for _, selected in selection.iterrows():

    audio_id = str(selected["audio_id"])
    path = ASR_DIR / f"asr_pilot_{audio_id}_H1-H5.csv"

    df = pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"},
        keep_default_na=False
    )

    assert len(df) == 5
    assert df["audio_id"].eq(audio_id).all()
    assert df["hypothesis_id"].tolist() == [
        "H1", "H2", "H3", "H4", "H5"
    ]

    record = {
        "audio_id": audio_id,
        "speaker": str(selected["speaker"])
    }

    for _, h in df.iterrows():
        record[h["hypothesis_id"]] = str(h["transcription"])

    records.append(record)

audit = pd.DataFrame(records)

assert len(audit) == 10
assert audit["audio_id"].is_unique

OUT.parent.mkdir(parents=True, exist_ok=True)
audit.to_csv(OUT, index=False, encoding="utf-8")

print("=== STEP 3.2 HYPOTHESIS CONTENT AUDIT ===")

for _, row in audit.iterrows():
    unique = len(set(row[h] for h in ["H1", "H2", "H3", "H4", "H5"]))

    print("\n" + "=" * 65)
    print("Audio ID:", row["audio_id"])
    print("Speaker:", row["speaker"])
    print("Raw unique:", unique)

    for hid in ["H1", "H2", "H3", "H4", "H5"]:
        print(f"{hid}: {row[hid]!r}")

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_pilot_content_audit_file"] = str(OUT)
progress["step3_pilot_content_audit_status"] = "generated_pending_review"
progress["next_task"] = "Review differences and freeze ASR configuration"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== CONTENT AUDIT SAVED ===")
print("Audio count:", len(audit))
print("Total hypotheses:", len(audit) * 5)
print("CSV:", OUT)
print("Checkpoint updated.")


In [ ]:

import io
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import matplotlib.pyplot as plt

from IPython.display import Audio, display

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
ASR_FILE = (
    PROJECT /
    "features/asr/pilot/asr_pilot_009600118_H1-H5.csv"
)
OUT = PROJECT / "data/metadata/asr_critical_listening_009600118.csv"
FIG = PROJECT / "results/figures/asr_critical_009600118.png"

TARGET_ID = "009600118"

pf = pq.ParquetFile(TRAIN)
matches = []

for group_id in range(pf.metadata.num_row_groups):
    table = pf.read_row_group(
        group_id, columns=["audio", "text", "speaker"]
    )

    audios = table.column("audio").to_pylist()
    texts = table.column("text").to_pylist()
    speakers = table.column("speaker").to_pylist()

    for audio, text, speaker in zip(
        audios, texts, speakers
    ):
        if Path(audio["path"]).stem == TARGET_ID:
            matches.append((audio["bytes"], text, speaker))

assert len(matches) == 1

audio_bytes, prompt_text, speaker = matches[0]

y, sr = sf.read(
    io.BytesIO(audio_bytes),
    dtype="float32"
)

assert sr == 16000 and y.ndim == 1
assert np.isfinite(y).all()

asr = pd.read_csv(
    ASR_FILE,
    dtype={"audio_id": "string"},
    keep_default_na=False
)

assert len(asr) == 5
assert asr["audio_id"].eq(TARGET_ID).all()

print("=== CRITICAL ASR AUDIO INSPECTION ===")
print("Audio ID:", TARGET_ID)
print("Speaker:", speaker)
print("Duration:", round(len(y) / sr, 3), "seconds")
print("Reading prompt:", repr(prompt_text))

print("\n=== ASR HYPOTHESES ===")
for _, row in asr.iterrows():
    print(row["hypothesis_id"], ":", repr(row["transcription"]))

print("\n=== ORIGINAL AUDIO ===")
display(Audio(y, rate=sr, normalize=False))

times = np.arange(len(y)) / sr

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(times, y, linewidth=0.6)
ax.set_title("Critical ASR Inspection: 009600118")
ax.set_xlabel("Time (seconds)")
ax.set_ylabel("Amplitude")
ax.set_xlim(0, len(y) / sr)
fig.tight_layout()

FIG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG, dpi=150)
plt.show()
plt.close(fig)

# Create a pending qualitative listening audit.
record = {
    "audio_id": TARGET_ID,
    "speaker": str(speaker),
    "reading_prompt": prompt_text,
    "manual_listening_observation": "pending",
    "audible_words_observation": "pending",
    "noise_or_recording_observation": "pending",
    "asr_error_observation": "pending",
    "manual_verbatim_transcript_available": False
}

OUT.parent.mkdir(parents=True, exist_ok=True)

if OUT.exists():
    previous = pd.read_csv(
        OUT,
        dtype={"audio_id": "string"},
        keep_default_na=False
    )
    assert len(previous) == 1
    assert previous.iloc[0]["audio_id"] == TARGET_ID
    print("\nExisting audit preserved:", OUT)
else:
    pd.DataFrame([record]).to_csv(
        OUT,
        index=False,
        encoding="utf-8"
    )
    print("\nAudit template saved:", OUT)

print("Waveform saved:", FIG)
print("=== INSPECTION READY ===")


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
AUDIT = PROJECT / "data/metadata/asr_critical_listening_009600118.csv"
REPORT = PROJECT / "reports/step03_critical_asr_review_009600118.md"
CHECK = PROJECT / "checkpoints/research_progress.json"

assert AUDIT.exists(), "Cell 18 audit file not found"
assert CHECK.exists(), "Research checkpoint not found"

df = pd.read_csv(
    AUDIT,
    dtype={"audio_id": "string", "speaker": "string"},
    keep_default_na=False
)

assert len(df) == 1
assert df.iloc[0]["audio_id"] == "009600118"

corrected_observation = (
    "Ey yet a possiple you are yen the wrong account."
)

df.loc[0, "manual_listening_observation"] = (
    "Pronunciation errors were audible."
)
df.loc[0, "audible_words_observation"] = corrected_observation
df.loc[0, "noise_or_recording_observation"] = (
    "Not assessed; no confirmed noise cause."
)
df.loc[0, "asr_error_observation"] = (
    "Substantial differences among H1-H5. "
    "H2 appeared closer to the researcher's "
    "approximate phonetic listening impression."
)
df.loc[0, "manual_verbatim_transcript_available"] = False
df["annotation_type"] = "informal_phonetic_listening_impression"
df["annotation_status"] = "reviewed"
df["updated_at"] = datetime.now(timezone.utc).isoformat()

df.to_csv(AUDIT, index=False, encoding="utf-8")

report = f"""# Critical ASR Listening Review

Audio ID: 009600118

## Researcher observation
Pronunciation errors were audible.

Corrected approximate phonetic listening impression:
{corrected_observation}

The researcher corrected a typing error:
'pissiple' -> 'possiple'.

This is not a verified verbatim transcript.

## ASR interpretation
H1-H5 produced substantially different transcripts.
H2 appeared closer to the researcher's listening impression.

No ground-truth WER or CER is reported.

## Methodological decisions
- Preserve all original ASR hypotheses.
- Do not treat reading prompts as verified transcripts.
- Do not select a winning hypothesis using this observation.
- Do not assert a definitive cause of the variability.
"""

REPORT.parent.mkdir(parents=True, exist_ok=True)
REPORT.write_text(report, encoding="utf-8")

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_critical_asr_audio"] = "009600118"
progress["step3_critical_asr_listening_status"] = "reviewed"
progress["step3_critical_asr_report"] = str(REPORT)
progress["next_task"] = (
    "Review and freeze ASR hypothesis generation design"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== CORRECTED ASR LISTENING AUDIT SAVED ===")
print("Audio ID:", df.iloc[0]["audio_id"])
print("Corrected observation:", corrected_observation)
print("Verified verbatim transcript:", False)
print("Audit:", AUDIT)
print("Report:", REPORT)
print("Next task:", progress["next_task"])
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

CONFIG_FILE = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
SUMMARY_FILE = PROJECT / "reports/step03_pilot_variability_summary.json"
AUDIT_FILE = PROJECT / "data/metadata/asr_critical_listening_009600118.csv"
CHECK_FILE = PROJECT / "checkpoints/research_progress.json"

FREEZE_REPORT = PROJECT / "reports/step03_asr_configuration_freeze.json"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

config = json.loads(CONFIG_FILE.read_text(encoding="utf-8"))
summary = json.loads(SUMMARY_FILE.read_text(encoding="utf-8"))

assert config["model_id"] == "openai/whisper-small.en"
assert config["model_revision"] == (
    "e8727524f962ee844a7319d92be39ac1bd25655a"
)
assert config["status"] in [
    "proposed_not_frozen", "frozen"
]

hypotheses = config["hypotheses"]
assert len(hypotheses) == 5

expected = {
    "H1": ("greedy", 1, False),
    "H2": ("beam_search", 5, False),
    "H3": ("sampling", None, True),
    "H4": ("sampling", None, True),
    "H5": ("sampling", None, True)
}

for h in hypotheses:
    hid = h["id"]
    method, beams, sampling = expected[hid]

    assert h["method"] == method
    assert h["do_sample"] == sampling

    if beams is not None:
        assert h["num_beams"] == beams

sampling = [h for h in hypotheses if h["method"] == "sampling"]

assert [h["seed"] for h in sampling] == [101, 202, 303]
assert all(h["temperature"] == 0.3 for h in sampling)

assert summary["total_audio"] == 10
assert summary["total_hypotheses"] == 50
assert summary["lexically_variable_audio_count"] == 5
assert summary["empty_transcriptions"] == 0
assert AUDIT_FILE.exists()

if config["status"] == "proposed_not_frozen":
    config["status"] = "frozen"
    config["freeze_date"] = datetime.now(timezone.utc).isoformat()

    config["freeze_rationale"] = (
        "Pilot of 10 recordings produced lexical variation "
        "in 5 recordings; decoding parameters retained "
        "for subsequent evaluation."
    )

    config["runtime_generation_rules"] = {
        "max_new_tokens": 128,
        "english_only_no_task_argument": True,
        "english_only_no_language_argument": True,
        "num_beams_sampling": 1,
        "dtype": "float16",
        "device": "cuda:0",
        "seed_reset_per_audio_and_hypothesis": True,
        "preserve_raw_transcriptions": True,
        "no_prompt_text_as_context": True,
        "no_fluency_labels_for_hypothesis_selection": True
    }

    CONFIG_FILE.write_text(
        json.dumps(config, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

report = {
    "status": "FROZEN",
    "model_id": config["model_id"],
    "model_revision": config["model_revision"],
    "hypotheses": hypotheses,
    "pilot_audio_count": 10,
    "pilot_hypothesis_count": 50,
    "pilot_lexically_variable_audio_count": 5,
    "configuration_sha256": sha256(CONFIG_FILE),
    "pilot_summary_sha256": sha256(SUMMARY_FILE),
    "listening_audit_sha256": sha256(AUDIT_FILE),
    "freeze_date": config["freeze_date"]
}

FREEZE_REPORT.parent.mkdir(parents=True, exist_ok=True)
FREEZE_REPORT.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

progress = json.loads(
    CHECK_FILE.read_text(encoding="utf-8")
)

progress["step3_asr_config_status"] = "frozen"
progress["step3_asr_config_sha256"] = sha256(CONFIG_FILE)
progress["step3_asr_freeze_report"] = str(FREEZE_REPORT)
progress["next_task"] = (
    "Generate H1-H5 ASR hypotheses for 400 pilot recordings"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK_FILE.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 3.2 ASR CONFIGURATION FROZEN ===")
print("Model:", config["model_id"])
print("Revision:", config["model_revision"])
print("Hypotheses:", len(hypotheses))
print("Pilot audio:", summary["total_audio"])
print(
    "Pilot lexical variability:",
    summary["lexically_variable_audio_count"]
)
print("Configuration status:", config["status"])
print("Configuration SHA256:", sha256(CONFIG_FILE))
print("Freeze report:", FREEZE_REPORT)
print("Next task:", progress["next_task"])
print("Checkpoint updated.")


In [ ]:

import io
import json
import time
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import torch

from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor
)

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT_DIR = PROJECT / "features/asr/full"
PART_DIR = OUT_DIR / "batch_0001-0100_parts"
FINAL = OUT_DIR / "asr_hypotheses_0001-0100.csv"

PART_DIR.mkdir(parents=True, exist_ok=True)

config = json.loads(CONFIG.read_text(encoding="utf-8"))

assert config["status"] == "frozen"
assert config["model_id"] == "openai/whisper-small.en"
assert config["model_revision"] == (
    "e8727524f962ee844a7319d92be39ac1bd25655a"
)

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

batch = pilot[
    pilot["pilot_index"].between(1, 100)
].sort_values("pilot_index").copy()

assert len(batch) == 100
assert batch["pilot_index"].tolist() == list(range(1, 101))
assert batch["audio_id"].is_unique

hypotheses = config["hypotheses"]

assert len(hypotheses) == 5
assert [h["id"] for h in hypotheses] == [
    "H1", "H2", "H3", "H4", "H5"
]

# Preserve frozen configuration integrity.
config_sha = hashlib.sha256(CONFIG.read_bytes()).hexdigest()

assert config_sha == (
    "1204e1414ec1332ee2f28caf076a799662d28828dadb8e361243140c388cfc77"
), "Frozen configuration has changed."

print("=== STEP 3.3: ASR BATCH 0001-0100 ===")
print("Model:", config["model_id"])
print("Audios:", len(batch))
print("Expected hypotheses:", len(batch) * len(hypotheses))
print("Config SHA256:", config_sha)

# Load original audio signals.
wanted = set(batch["audio_id"])
audio_data = {}

pf = pq.ParquetFile(TRAIN)

for group_id in range(pf.metadata.num_row_groups):
    table = pf.read_row_group(
        group_id,
        columns=["audio"]
    )

    for audio in table.column("audio").to_pylist():
        audio_id = Path(audio["path"]).stem

        if audio_id in wanted:
            assert audio_id not in audio_data
            audio_data[audio_id] = audio["bytes"]

assert set(audio_data) == wanted

device = "cuda:0"
assert torch.cuda.is_available()

processor = WhisperProcessor.from_pretrained(
    config["model_id"],
    revision=config["model_revision"]
)

model = WhisperForConditionalGeneration.from_pretrained(
    config["model_id"],
    revision=config["model_revision"],
    dtype=torch.float16
).to(device)

model.eval()

expected_columns = [
    "pilot_index", "audio_id", "speaker",
    "hypothesis_id", "method",
    "model_id", "model_revision",
    "do_sample", "num_beams", "temperature",
    "seed", "transcription", "duration_sec",
    "inference_sec"
]

def validate_part(path, row):
    df = pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"},
        keep_default_na=False
    )

    assert len(df) == 5
    assert df.columns.tolist() == expected_columns
    assert df["pilot_index"].eq(int(row["pilot_index"])).all()
    assert df["audio_id"].eq(str(row["audio_id"])).all()
    assert df["speaker"].eq(str(row["speaker"])).all()
    assert df["hypothesis_id"].tolist() == [
        "H1", "H2", "H3", "H4", "H5"
    ]
    assert df["model_id"].eq(config["model_id"]).all()
    assert df["model_revision"].eq(config["model_revision"]).all()
    return df

# Generate and checkpoint one audio at a time.
for n, (_, row) in enumerate(batch.iterrows(), start=1):

    audio_id = str(row["audio_id"])
    part = PART_DIR / f"asr_{audio_id}_H1-H5.csv"

    if part.exists():
        validate_part(part, row)
        print(f"[{n:03d}/100] SKIP {audio_id}")
        continue

    y, sr = sf.read(
        io.BytesIO(audio_data[audio_id]),
        dtype="float32"
    )

    assert sr == 16000 and y.ndim == 1
    assert np.isfinite(y).all()

    inputs = processor(
        y,
        sampling_rate=sr,
        return_tensors="pt"
    )

    input_features = inputs.input_features.to(
        device=device,
        dtype=torch.float16
    )

    records = []

    for h in hypotheses:

        kwargs = {
            "do_sample": False,
            "num_beams": 1,
            "max_new_tokens": 128
        }

        seed = None
        temperature = None

        if h["method"] == "beam_search":
            kwargs["num_beams"] = int(h["num_beams"])

        elif h["method"] == "sampling":
            seed = int(h["seed"])
            temperature = float(h["temperature"])

            kwargs["do_sample"] = True
            kwargs["temperature"] = temperature

            torch.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)

        torch.cuda.synchronize()
        start = time.perf_counter()

        with torch.inference_mode():
            generated_ids = model.generate(
                input_features,
                **kwargs
            )

        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start

        transcription = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )[0].strip()

        records.append({
            "pilot_index": int(row["pilot_index"]),
            "audio_id": audio_id,
            "speaker": str(row["speaker"]),
            "hypothesis_id": h["id"],
            "method": h["method"],
            "model_id": config["model_id"],
            "model_revision": config["model_revision"],
            "do_sample": kwargs["do_sample"],
            "num_beams": kwargs["num_beams"],
            "temperature": temperature,
            "seed": seed,
            "transcription": transcription,
            "duration_sec": len(y) / sr,
            "inference_sec": elapsed
        })

    result = pd.DataFrame(records, columns=expected_columns)

    # Atomic checkpoint write.
    tmp = part.with_suffix(".tmp")
    result.to_csv(tmp, index=False, encoding="utf-8")
    tmp.replace(part)

    validate_part(part, row)

    if n % 10 == 0 or n == 1:
        print(f"[{n:03d}/100] COMPLETE {audio_id}")

# Combine verified per-audio checkpoints.
pieces = []

for _, row in batch.iterrows():
    audio_id = str(row["audio_id"])
    part = PART_DIR / f"asr_{audio_id}_H1-H5.csv"
    pieces.append(validate_part(part, row))

combined = pd.concat(pieces, ignore_index=True)

assert len(combined) == 500
assert combined["audio_id"].nunique() == 100

temp_final = FINAL.with_suffix(".tmp")
combined.to_csv(temp_final, index=False, encoding="utf-8")
temp_final.replace(FINAL)

sha = hashlib.sha256(FINAL.read_bytes()).hexdigest()

unique_counts = combined.groupby(
    "audio_id"
)["transcription"].nunique()

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_full_batch_0001-0100"] = "completed_pending_qc"
progress["step3_full_batch_0001-0100_sha256"] = sha
progress["step3_full_generated_audio_count"] = 100
progress["step3_full_generated_hypothesis_count"] = 500
progress["next_task"] = "Quality control for ASR batch 0001-0100"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== ASR BATCH 0001-0100 SUMMARY ===")
print("Audios:", combined["audio_id"].nunique())
print("Hypotheses:", len(combined))
print("Unique transcripts mean:", round(unique_counts.mean(), 3))
print("Audios with raw variability:", int((unique_counts > 1).sum()))
print(
    "Empty transcriptions:",
    int(combined["transcription"].eq("").sum())
)
print("CSV:", FINAL)
print("SHA256:", sha)
print("Checkpoint updated.")


In [ ]:

import json
import re
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
ASR = PROJECT / "features/asr/full/asr_hypotheses_0001-0100.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

QC_REPORT = PROJECT / "reports/step03_batch_0001-0100_qc.json"
QC_TABLE = PROJECT / "reports/step03_batch_0001-0100_qc_per_audio.csv"

EXPECTED_CONFIG_SHA = (
    "1204e1414ec1332ee2f28caf076a799662d28828dadb8e361243140c388cfc77"
)
EXPECTED_ASR_SHA = (
    "b33b7d0e2564c43738a5f254221e639d53b72edf6f5ced0b407962c152e02fb3"
)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(CONFIG) == EXPECTED_CONFIG_SHA
assert sha256(ASR) == EXPECTED_ASR_SHA

config = json.loads(CONFIG.read_text(encoding="utf-8"))
assert config["status"] == "frozen"

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

expected = pilot.loc[
    pilot["pilot_index"].between(1, 100),
    ["pilot_index", "audio_id", "speaker"]
].sort_values("pilot_index")

df = pd.read_csv(
    ASR,
    dtype={"audio_id": "string", "speaker": "string"},
    keep_default_na=False
)

assert len(expected) == 100
assert len(df) == 500
assert df["audio_id"].nunique() == 100

hypothesis_ids = ["H1", "H2", "H3", "H4", "H5"]

assert not df.duplicated(
    ["audio_id", "hypothesis_id"]
).any()

assert df["model_id"].eq(config["model_id"]).all()
assert df["model_revision"].eq(config["model_revision"]).all()

def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s']", " ", text)
    return re.sub(r"\s+", " ", text).strip()

qc_rows = []

for _, item in expected.iterrows():

    audio_id = str(item["audio_id"])

    subset = df.loc[
        df["audio_id"] == audio_id
    ].sort_values("hypothesis_id")

    assert len(subset) == 5
    assert subset["hypothesis_id"].tolist() == hypothesis_ids
    assert subset["pilot_index"].eq(
        int(item["pilot_index"])
    ).all()
    assert subset["speaker"].eq(str(item["speaker"])).all()

    normalized = subset["transcription"].map(normalize_text)

    qc_rows.append({
        "pilot_index": int(item["pilot_index"]),
        "audio_id": audio_id,
        "speaker": str(item["speaker"]),
        "raw_unique": int(subset["transcription"].nunique()),
        "lexical_unique": int(normalized.nunique()),
        "empty_transcriptions": int(
            subset["transcription"].str.strip().eq("").sum()
        ),
        "max_transcription_chars": int(
            subset["transcription"].str.len().max()
        ),
        "duration_sec": float(subset["duration_sec"].iloc[0])
    })

qc = pd.DataFrame(qc_rows)

assert len(qc) == 100
assert int(qc["empty_transcriptions"].sum()) == 0
assert (qc["duration_sec"] > 0).all()

summary = {
    "batch": "0001-0100",
    "status": "PASS",
    "audio_count": 100,
    "hypothesis_count": 500,
    "raw_variable_audio_count": int(
        (qc["raw_unique"] > 1).sum()
    ),
    "lexically_variable_audio_count": int(
        (qc["lexical_unique"] > 1).sum()
    ),
    "mean_raw_unique": float(qc["raw_unique"].mean()),
    "mean_lexical_unique": float(qc["lexical_unique"].mean()),
    "empty_transcriptions": int(
        qc["empty_transcriptions"].sum()
    ),
    "config_sha256": sha256(CONFIG),
    "asr_csv_sha256": sha256(ASR),
    "created_at": datetime.now(timezone.utc).isoformat()
}

QC_REPORT.parent.mkdir(parents=True, exist_ok=True)

QC_TABLE_TMP = QC_TABLE.with_suffix(".tmp")
qc.to_csv(QC_TABLE_TMP, index=False, encoding="utf-8")
QC_TABLE_TMP.replace(QC_TABLE)

QC_REPORT.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_full_batch_0001-0100"] = "qc_pass"
progress["step3_full_batch_0001-0100_qc_report"] = str(QC_REPORT)
progress["next_task"] = "Create Batch 1 backup, then Batch 2"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 3.3 BATCH 1 QC SUMMARY ===")
for key, value in summary.items():
    print(f"{key}: {value}")

print("\n=== VARIABILITY DISTRIBUTION ===")
print(
    qc["lexical_unique"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nQC report:", QC_REPORT)
print("QC table:", QC_TABLE)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
import zipfile
from pathlib import Path
from datetime import datetime, timezone

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
BACKUP = Path("/kaggle/working/asr_fluency_step03_batch_0001-0100.zip")
CHECK = PROJECT / "checkpoints/research_progress.json"

EXPECTED_CONFIG_SHA = "1204e1414ec1332ee2f28caf076a799662d28828dadb8e361243140c388cfc77"
EXPECTED_ASR_SHA = "b33b7d0e2564c43738a5f254221e639d53b72edf6f5ced0b407962c152e02fb3"

def file_sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

required_files = [
    "configs/asr_hypothesis_pilot_v1.json",
    "data/metadata/pilot_selection_400.csv",
    "data/metadata/asr_pilot_selection_10.csv",
    "data/metadata/asr_critical_listening_009600118.csv",
    "features/asr/full/asr_hypotheses_0001-0100.csv",
    "reports/step03_asr_configuration_freeze.json",
    "reports/step03_pilot_variability_summary.json",
    "reports/step03_pilot_variability_per_audio.csv",
    "reports/step03_pilot_hypothesis_content_audit.csv",
    "reports/step03_critical_asr_review_009600118.md",
    "reports/step03_batch_0001-0100_qc.json",
    "reports/step03_batch_0001-0100_qc_per_audio.csv",
    "checkpoints/research_progress.json",
]

for relative in required_files:
    assert (PROJECT / relative).is_file(), (
        f"Required file missing: {relative}"
    )

assert file_sha256(
    PROJECT / "configs/asr_hypothesis_pilot_v1.json"
) == EXPECTED_CONFIG_SHA

assert file_sha256(
    PROJECT / "features/asr/full/asr_hypotheses_0001-0100.csv"
) == EXPECTED_ASR_SHA

qc = json.loads(
    (PROJECT / "reports/step03_batch_0001-0100_qc.json")
    .read_text(encoding="utf-8")
)

assert qc["status"] == "PASS"
assert qc["audio_count"] == 100
assert qc["hypothesis_count"] == 500
assert qc["config_sha256"] == EXPECTED_CONFIG_SHA
assert qc["asr_csv_sha256"] == EXPECTED_ASR_SHA

part_dir = PROJECT / "features/asr/full/batch_0001-0100_parts"
part_files = sorted(part_dir.glob("asr_*_H1-H5.csv"))
assert len(part_files) == 100, (
    f"Expected 100 audio checkpoints, found {len(part_files)}"
)

# Update checkpoint before packaging it.
progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_full_batch_0001-0100"] = "qc_pass"
progress["step3_batch1_backup_status"] = "packaging"
progress["step3_batch1_backup_filename"] = BACKUP.name
progress["next_task"] = (
    "Restore/retain Batch 1 backup and start ASR Batch 0101-0200"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# Include the previous smoke tests, pilot ASR outputs,
# and all per-audio checkpoints.
files = [PROJECT / p for p in required_files]

for directory in [
    PROJECT / "features/asr/pilot",
    PROJECT / "features/asr/full/batch_0001-0100_parts",
]:
    files.extend(
        p for p in directory.rglob("*") if p.is_file()
    )

files.extend(
    PROJECT.glob("features/asr/asr_smoke_test_*.csv")
)

for name in [
    "reports/step03_whisper_smoke_test.json",
    "reports/step03_whisper_beam_test.json",
    "reports/step03_sampling_smoke_test.json",
]:
    p = PROJECT / name
    if p.is_file():
        files.append(p)

files = sorted(set(files))
assert len(files) >= 113

manifest = {
    "backup_name": BACKUP.name,
    "batch": "0001-0100",
    "qc_status": "PASS",
    "audio_count": 100,
    "hypothesis_count": 500,
    "project_root": "acoustic_linguistic_research",
    "raw_parquet_included": False,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "files": {}
}

temporary = BACKUP.with_suffix(".zip.tmp")

with zipfile.ZipFile(
    temporary, "w", compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as archive:

    for path in files:
        relative = path.relative_to(PROJECT).as_posix()
        archive_name = f"acoustic_linguistic_research/{relative}"
        manifest["files"][archive_name] = file_sha256(path)
        archive.write(path, arcname=archive_name)

    archive.writestr(
        "acoustic_linguistic_research/"
        "backup_manifest_batch_0001-0100.json",
        json.dumps(manifest, indent=2, ensure_ascii=False)
    )

temporary.replace(BACKUP)

# Independently verify the ZIP contents.
with zipfile.ZipFile(BACKUP, "r") as archive:
    assert archive.testzip() is None

    for name, expected_hash in manifest["files"].items():
        actual_hash = hashlib.sha256(
            archive.read(name)
        ).hexdigest()

        assert actual_hash == expected_hash, (
            f"ZIP integrity failure: {name}"
        )

print("=== STEP 3.3 BATCH 1 BACKUP VERIFIED ===")
print("Backup:", BACKUP)
print("Files archived:", len(files))
print("Audio checkpoint files:", len(part_files))
print("ZIP size MB:", round(BACKUP.stat().st_size / 1024**2, 3))
print("ZIP SHA256:", file_sha256(BACKUP))
print("QC status:", qc["status"])
print("Archive integrity: PASS")
print("Next batch: 0101-0200")


In [ ]:

import io
import json
import time
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import torch

from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor
)

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT_DIR = PROJECT / "features/asr/full"
PART_DIR = OUT_DIR / "batch_0101-0200_parts"
FINAL = OUT_DIR / "asr_hypotheses_0101-0200.csv"

PART_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_CONFIG_SHA = (
    "1204e1414ec1332ee2f28caf076a799662d28828dadb8e361243140c388cfc77"
)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(CONFIG) == EXPECTED_CONFIG_SHA, (
    "Frozen ASR configuration has changed."
)

config = json.loads(CONFIG.read_text(encoding="utf-8"))

assert config["status"] == "frozen"
assert config["model_id"] == "openai/whisper-small.en"
assert config["model_revision"] == (
    "e8727524f962ee844a7319d92be39ac1bd25655a"
)

# Verify that Batch 1 is complete and unchanged.
BATCH1 = OUT_DIR / "asr_hypotheses_0001-0100.csv"
QC1 = PROJECT / "reports/step03_batch_0001-0100_qc.json"

assert QC1.exists() and BATCH1.exists()

qc1 = json.loads(QC1.read_text(encoding="utf-8"))
assert qc1["status"] == "PASS"
assert sha256(BATCH1) == qc1["asr_csv_sha256"]

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

batch = pilot.loc[
    pilot["pilot_index"].between(101, 200)
].sort_values("pilot_index").copy()

assert len(batch) == 100
assert batch["pilot_index"].tolist() == list(range(101, 201))
assert batch["audio_id"].is_unique
assert batch["speaker"].notna().all()

hypotheses = config["hypotheses"]

assert len(hypotheses) == 5
assert [h["id"] for h in hypotheses] == [
    "H1", "H2", "H3", "H4", "H5"
]

expected_columns = [
    "pilot_index", "audio_id", "speaker",
    "hypothesis_id", "method",
    "model_id", "model_revision",
    "do_sample", "num_beams", "temperature",
    "seed", "transcription", "duration_sec",
    "inference_sec"
]

def validate_part(path, row):
    df = pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"},
        keep_default_na=False
    )

    assert len(df) == 5
    assert df.columns.tolist() == expected_columns
    assert df["pilot_index"].eq(int(row["pilot_index"])).all()
    assert df["audio_id"].eq(str(row["audio_id"])).all()
    assert df["speaker"].eq(str(row["speaker"])).all()
    assert df["hypothesis_id"].tolist() == [
        "H1", "H2", "H3", "H4", "H5"
    ]
    assert df["model_id"].eq(config["model_id"]).all()
    assert df["model_revision"].eq(config["model_revision"]).all()

    for _, record in df.iterrows():
        spec = next(
            h for h in hypotheses
            if h["id"] == record["hypothesis_id"]
        )

        assert record["method"] == spec["method"]
        assert str(record["transcription"]) != ""

        if spec["method"] == "sampling":
            assert str(record["do_sample"]).lower() == "true"
            assert int(record["num_beams"]) == 1
            assert np.isclose(
                float(record["temperature"]),
                float(spec["temperature"])
            )
            assert int(record["seed"]) == int(spec["seed"])
        else:
            assert str(record["do_sample"]).lower() == "false"
            expected_beams = (
                int(spec["num_beams"])
                if spec["method"] == "beam_search"
                else 1
            )
            assert int(record["num_beams"]) == expected_beams

        assert np.isfinite(float(record["duration_sec"]))
        assert float(record["duration_sec"]) > 0

    return df

# Identify completed audio checkpoints before model loading.
pending = []
completed_before_run = 0

for _, row in batch.iterrows():
    part = PART_DIR / f"asr_{row['audio_id']}_H1-H5.csv"

    if part.exists():
        validate_part(part, row)
        completed_before_run += 1
    else:
        pending.append(row)

print("=== STEP 3.3: ASR BATCH 0101-0200 ===")
print("Model:", config["model_id"])
print("Revision:", config["model_revision"])
print("Batch range: 101-200")
print("Total audios:", len(batch))
print("Completed checkpoints:", completed_before_run)
print("Pending audios:", len(pending))
print("Expected hypotheses:", 500)
print("Frozen config SHA256:", sha256(CONFIG))

if pending:
    assert TRAIN.exists(), "Raw train Parquet missing"
    assert torch.cuda.is_available(), "CUDA GPU unavailable"

    wanted = {str(row["audio_id"]) for row in pending}
    audio_data = {}

    pf = pq.ParquetFile(TRAIN)

    assert "audio" in pf.schema_arrow.names

    for group_id in range(pf.metadata.num_row_groups):
        table = pf.read_row_group(
            group_id,
            columns=["audio"]
        )

        for audio in table.column("audio").to_pylist():
            audio_id = Path(audio["path"]).stem

            if audio_id in wanted:
                assert audio_id not in audio_data
                audio_data[audio_id] = audio["bytes"]

    assert set(audio_data) == wanted

    device = "cuda:0"

    processor = WhisperProcessor.from_pretrained(
        config["model_id"],
        revision=config["model_revision"]
    )

    model = WhisperForConditionalGeneration.from_pretrained(
        config["model_id"],
        revision=config["model_revision"],
        dtype=torch.float16
    ).to(device)

    model.eval()

    for count, row in enumerate(pending, start=1):
        audio_id = str(row["audio_id"])
        pilot_index = int(row["pilot_index"])

        y, sr = sf.read(
            io.BytesIO(audio_data[audio_id]),
            dtype="float32"
        )

        assert sr == 16000 and y.ndim == 1
        assert np.isfinite(y).all()

        inputs = processor(
            y,
            sampling_rate=sr,
            return_tensors="pt"
        )

        input_features = inputs.input_features.to(
            device=device,
            dtype=torch.float16
        )

        records = []

        for h in hypotheses:
            kwargs = {
                "do_sample": False,
                "num_beams": 1,
                "max_new_tokens": 128
            }

            seed = None
            temperature = None

            if h["method"] == "beam_search":
                kwargs["num_beams"] = int(h["num_beams"])

            elif h["method"] == "sampling":
                seed = int(h["seed"])
                temperature = float(h["temperature"])

                kwargs["do_sample"] = True
                kwargs["temperature"] = temperature

                torch.manual_seed(seed)
                torch.cuda.manual_seed_all(seed)

            torch.cuda.synchronize()
            start = time.perf_counter()

            with torch.inference_mode():
                generated_ids = model.generate(
                    input_features,
                    **kwargs
                )

            torch.cuda.synchronize()
            elapsed = time.perf_counter() - start

            transcription = processor.batch_decode(
                generated_ids,
                skip_special_tokens=True
            )[0].strip()

            records.append({
                "pilot_index": pilot_index,
                "audio_id": audio_id,
                "speaker": str(row["speaker"]),
                "hypothesis_id": h["id"],
                "method": h["method"],
                "model_id": config["model_id"],
                "model_revision": config["model_revision"],
                "do_sample": kwargs["do_sample"],
                "num_beams": kwargs["num_beams"],
                "temperature": temperature,
                "seed": seed,
                "transcription": transcription,
                "duration_sec": len(y) / sr,
                "inference_sec": elapsed
            })

        result = pd.DataFrame(
            records,
            columns=expected_columns
        )

        part = PART_DIR / f"asr_{audio_id}_H1-H5.csv"
        tmp = part.with_suffix(".tmp")

        result.to_csv(
            tmp,
            index=False,
            encoding="utf-8"
        )
        tmp.replace(part)

        validate_part(part, row)

        if count == 1 or count % 10 == 0 or count == len(pending):
            print(
                f"[{count:03d}/{len(pending):03d}] "
                f"COMPLETE index={pilot_index} audio={audio_id}"
            )

# Final validation and assembly.
pieces = []

for _, row in batch.iterrows():
    part = PART_DIR / f"asr_{row['audio_id']}_H1-H5.csv"
    pieces.append(validate_part(part, row))

combined = pd.concat(pieces, ignore_index=True)

assert len(combined) == 500
assert combined["audio_id"].nunique() == 100
assert combined["pilot_index"].nunique() == 100
assert not combined.duplicated(
    ["audio_id", "hypothesis_id"]
).any()

tmp_final = FINAL.with_suffix(".tmp")

combined.to_csv(
    tmp_final,
    index=False,
    encoding="utf-8"
)
tmp_final.replace(FINAL)

final_sha = sha256(FINAL)

unique_counts = combined.groupby(
    "audio_id"
)["transcription"].nunique()

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)

progress["step3_full_batch_0101-0200"] = "completed_pending_qc"
progress["step3_full_batch_0101-0200_sha256"] = final_sha
progress["step3_full_generated_audio_count"] = 200
progress["step3_full_generated_hypothesis_count"] = 1000
progress["next_task"] = (
    "Quality control for ASR batch 0101-0200"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== ASR BATCH 0101-0200 SUMMARY ===")
print("Audios:", combined["audio_id"].nunique())
print("Hypotheses:", len(combined))
print("Mean unique transcripts:", round(unique_counts.mean(), 3))
print("Audios with raw variability:", int((unique_counts > 1).sum()))
print(
    "Empty transcriptions:",
    int(combined["transcription"].str.strip().eq("").sum())
)
print("CSV:", FINAL)
print("SHA256:", final_sha)
print("Checkpoint updated.")


In [ ]:

import io
import json
import time
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import torch

from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor
)

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT_DIR = PROJECT / "features/asr/full"
PART_DIR = OUT_DIR / "batch_0101-0200_parts"
FINAL = OUT_DIR / "asr_hypotheses_0101-0200.csv"

PART_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_CONFIG_SHA = (
    "1204e1414ec1332ee2f28caf076a799662d28828dadb8e361243140c388cfc77"
)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(CONFIG) == EXPECTED_CONFIG_SHA

config = json.loads(CONFIG.read_text(encoding="utf-8"))

assert config["status"] == "frozen"
assert config["model_id"] == "openai/whisper-small.en"
assert config["model_revision"] == (
    "e8727524f962ee844a7319d92be39ac1bd25655a"
)

# Verify Batch 1 remains unchanged.
BATCH1 = OUT_DIR / "asr_hypotheses_0001-0100.csv"
QC1 = PROJECT / "reports/step03_batch_0001-0100_qc.json"

qc1 = json.loads(QC1.read_text(encoding="utf-8"))
assert qc1["status"] == "PASS"
assert sha256(BATCH1) == qc1["asr_csv_sha256"]

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

batch = pilot.loc[
    pilot["pilot_index"].between(101, 200)
].sort_values("pilot_index").copy()

assert len(batch) == 100
assert batch["pilot_index"].tolist() == list(range(101, 201))
assert batch["audio_id"].is_unique

hypotheses = config["hypotheses"]

assert [h["id"] for h in hypotheses] == [
    "H1", "H2", "H3", "H4", "H5"
]

expected_columns = [
    "pilot_index", "audio_id", "speaker",
    "hypothesis_id", "method",
    "model_id", "model_revision",
    "do_sample", "num_beams", "temperature",
    "seed", "transcription", "duration_sec",
    "inference_sec"
]

def validate_part(path, row):
    df = pd.read_csv(
        path,
        dtype={
            "audio_id": "string",
            "speaker": "string",
            "seed": "string"
        },
        keep_default_na=False
    )

    assert len(df) == 5
    assert df.columns.tolist() == expected_columns
    assert df["pilot_index"].eq(int(row["pilot_index"])).all()
    assert df["audio_id"].eq(str(row["audio_id"])).all()
    assert df["speaker"].eq(str(row["speaker"])).all()
    assert df["hypothesis_id"].tolist() == [
        "H1", "H2", "H3", "H4", "H5"
    ]
    assert df["model_id"].eq(config["model_id"]).all()
    assert df["model_revision"].eq(config["model_revision"]).all()

    for _, record in df.iterrows():
        spec = next(
            h for h in hypotheses
            if h["id"] == record["hypothesis_id"]
        )

        assert record["method"] == spec["method"]
        assert isinstance(record["transcription"], str)

        if spec["method"] == "sampling":
            assert str(record["do_sample"]).lower() == "true"
            assert int(record["num_beams"]) == 1
            assert np.isclose(
                float(record["temperature"]),
                float(spec["temperature"])
            )
            # CSV may contain 101.0 instead of 101.
            assert float(record["seed"]) == float(spec["seed"])
        else:
            assert str(record["do_sample"]).lower() == "false"

            expected_beams = (
                int(spec["num_beams"])
                if spec["method"] == "beam_search"
                else 1
            )

            assert int(record["num_beams"]) == expected_beams

        assert np.isfinite(float(record["duration_sec"]))
        assert float(record["duration_sec"]) > 0

    return df

# Discover completed checkpoints before loading the model.
pending = []
completed_before_run = 0

for _, row in batch.iterrows():
    part = PART_DIR / f"asr_{row['audio_id']}_H1-H5.csv"

    if part.exists():
        validate_part(part, row)
        completed_before_run += 1
    else:
        pending.append(row)

print("=== STEP 3.3 BATCH 2 RESUME ===")
print("Total audios:", len(batch))
print("Completed checkpoints:", completed_before_run)
print("Pending audios:", len(pending))
print("Frozen config SHA256:", sha256(CONFIG))

if pending:
    assert TRAIN.exists()
    assert torch.cuda.is_available()

    wanted = {str(row["audio_id"]) for row in pending}
    audio_data = {}

    pf = pq.ParquetFile(TRAIN)
    assert "audio" in pf.schema_arrow.names

    for group_id in range(pf.metadata.num_row_groups):
        table = pf.read_row_group(
            group_id,
            columns=["audio"]
        )

        for audio in table.column("audio").to_pylist():
            audio_id = Path(audio["path"]).stem

            if audio_id in wanted:
                assert audio_id not in audio_data
                audio_data[audio_id] = audio["bytes"]

    assert set(audio_data) == wanted

    device = "cuda:0"

    processor = WhisperProcessor.from_pretrained(
        config["model_id"],
        revision=config["model_revision"]
    )

    model = WhisperForConditionalGeneration.from_pretrained(
        config["model_id"],
        revision=config["model_revision"],
        dtype=torch.float16
    ).to(device)

    model.eval()

    for count, row in enumerate(pending, start=1):
        audio_id = str(row["audio_id"])
        pilot_index = int(row["pilot_index"])

        y, sr = sf.read(
            io.BytesIO(audio_data[audio_id]),
            dtype="float32"
        )

        assert sr == 16000 and y.ndim == 1
        assert np.isfinite(y).all()

        inputs = processor(
            y,
            sampling_rate=sr,
            return_tensors="pt"
        )

        input_features = inputs.input_features.to(
            device=device,
            dtype=torch.float16
        )

        records = []

        for h in hypotheses:
            kwargs = {
                "do_sample": False,
                "num_beams": 1,
                "max_new_tokens": 128
            }

            seed = None
            temperature = None

            if h["method"] == "beam_search":
                kwargs["num_beams"] = int(h["num_beams"])

            elif h["method"] == "sampling":
                seed = int(h["seed"])
                temperature = float(h["temperature"])

                kwargs["do_sample"] = True
                kwargs["temperature"] = temperature

                torch.manual_seed(seed)
                torch.cuda.manual_seed_all(seed)

            torch.cuda.synchronize()
            start = time.perf_counter()

            with torch.inference_mode():
                generated_ids = model.generate(
                    input_features,
                    **kwargs
                )

            torch.cuda.synchronize()
            elapsed = time.perf_counter() - start

            transcription = processor.batch_decode(
                generated_ids,
                skip_special_tokens=True
            )[0].strip()

            records.append({
                "pilot_index": pilot_index,
                "audio_id": audio_id,
                "speaker": str(row["speaker"]),
                "hypothesis_id": h["id"],
                "method": h["method"],
                "model_id": config["model_id"],
                "model_revision": config["model_revision"],
                "do_sample": kwargs["do_sample"],
                "num_beams": kwargs["num_beams"],
                "temperature": temperature,
                "seed": seed,
                "transcription": transcription,
                "duration_sec": len(y) / sr,
                "inference_sec": elapsed
            })

        result = pd.DataFrame(
            records,
            columns=expected_columns
        )

        part = PART_DIR / f"asr_{audio_id}_H1-H5.csv"
        tmp = part.with_suffix(".tmp")

        result.to_csv(
            tmp,
            index=False,
            encoding="utf-8"
        )
        tmp.replace(part)

        validate_part(part, row)

        if count == 1 or count % 10 == 0 or count == len(pending):
            print(
                f"[{count:03d}/{len(pending):03d}] "
                f"COMPLETE index={pilot_index} audio={audio_id}"
            )

pieces = []

for _, row in batch.iterrows():
    part = PART_DIR / f"asr_{row['audio_id']}_H1-H5.csv"
    pieces.append(validate_part(part, row))

combined = pd.concat(pieces, ignore_index=True)

assert len(combined) == 500
assert combined["audio_id"].nunique() == 100
assert combined["pilot_index"].nunique() == 100
assert not combined.duplicated(
    ["audio_id", "hypothesis_id"]
).any()

tmp_final = FINAL.with_suffix(".tmp")
combined.to_csv(tmp_final, index=False, encoding="utf-8")
tmp_final.replace(FINAL)

final_sha = sha256(FINAL)

unique_counts = combined.groupby(
    "audio_id"
)["transcription"].nunique()

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)

progress["step3_full_batch_0101-0200"] = "completed_pending_qc"
progress["step3_full_batch_0101-0200_sha256"] = final_sha
progress["step3_full_generated_audio_count"] = 200
progress["step3_full_generated_hypothesis_count"] = 1000
progress["next_task"] = "Quality control for ASR batch 0101-0200"
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n=== ASR BATCH 0101-0200 SUMMARY ===")
print("Audios:", combined["audio_id"].nunique())
print("Hypotheses:", len(combined))
print("Mean unique transcripts:", round(unique_counts.mean(), 3))
print("Audios with raw variability:", int((unique_counts > 1).sum()))
print(
    "Empty transcriptions:",
    int(combined["transcription"].str.strip().eq("").sum())
)
print("CSV:", FINAL)
print("SHA256:", final_sha)
print("Checkpoint updated.")


In [ ]:

import json
import re
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
ASR1 = PROJECT / "features/asr/full/asr_hypotheses_0001-0100.csv"
ASR2 = PROJECT / "features/asr/full/asr_hypotheses_0101-0200.csv"

QC1 = PROJECT / "reports/step03_batch_0001-0100_qc.json"
QC2 = PROJECT / "reports/step03_batch_0101-0200_qc.json"
TABLE2 = PROJECT / "reports/step03_batch_0101-0200_qc_per_audio.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"

CONFIG_SHA = "1204e1414ec1332ee2f28caf076a799662d28828dadb8e361243140c388cfc77"
ASR2_SHA = "e838bb9b722c73aeef2cf10dee6e18ea0970e6df29f4c0e5bcf37b142f52595f"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(CONFIG) == CONFIG_SHA
assert sha256(ASR2) == ASR2_SHA

config = json.loads(CONFIG.read_text(encoding="utf-8"))
assert config["status"] == "frozen"

# Confirm Batch 1 integrity remains intact.
qc1 = json.loads(QC1.read_text(encoding="utf-8"))
assert qc1["status"] == "PASS"
assert sha256(ASR1) == qc1["asr_csv_sha256"]

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

expected = pilot.loc[
    pilot["pilot_index"].between(101, 200),
    ["pilot_index", "audio_id", "speaker"]
].sort_values("pilot_index")

df = pd.read_csv(
    ASR2,
    dtype={
        "audio_id": "string",
        "speaker": "string",
        "seed": "string"
    },
    keep_default_na=False
)

assert len(expected) == 100
assert len(df) == 500
assert df["audio_id"].nunique() == 100
assert df["pilot_index"].nunique() == 100

hypotheses = config["hypotheses"]
specs = {h["id"]: h for h in hypotheses}

assert set(specs) == {"H1", "H2", "H3", "H4", "H5"}

assert not df.duplicated(
    ["audio_id", "hypothesis_id"]
).any()

assert df["model_id"].eq(config["model_id"]).all()
assert df["model_revision"].eq(config["model_revision"]).all()

def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s']", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

qc_rows = []

for _, item in expected.iterrows():
    audio_id = str(item["audio_id"])

    subset = df.loc[
        df["audio_id"] == audio_id
    ].sort_values("hypothesis_id")

    assert len(subset) == 5
    assert subset["hypothesis_id"].tolist() == [
        "H1", "H2", "H3", "H4", "H5"
    ]

    assert subset["pilot_index"].eq(
        int(item["pilot_index"])
    ).all()

    assert subset["speaker"].eq(
        str(item["speaker"])
    ).all()

    for _, record in subset.iterrows():
        spec = specs[record["hypothesis_id"]]

        assert record["method"] == spec["method"]

        if spec["method"] == "sampling":
            assert str(record["do_sample"]).lower() == "true"
            assert int(record["num_beams"]) == 1
            assert np.isclose(
                float(record["temperature"]),
                float(spec["temperature"])
            )
            assert float(record["seed"]) == float(spec["seed"])
        else:
            assert str(record["do_sample"]).lower() == "false"
            expected_beams = (
                int(spec["num_beams"])
                if spec["method"] == "beam_search"
                else 1
            )
            assert int(record["num_beams"]) == expected_beams

        assert np.isfinite(float(record["duration_sec"]))
        assert float(record["duration_sec"]) > 0

    normalized = subset["transcription"].map(normalize_text)

    qc_rows.append({
        "pilot_index": int(item["pilot_index"]),
        "audio_id": audio_id,
        "speaker": str(item["speaker"]),
        "raw_unique": int(subset["transcription"].nunique()),
        "lexical_unique": int(normalized.nunique()),
        "empty_transcriptions": int(
            subset["transcription"].str.strip().eq("").sum()
        ),
        "max_transcription_chars": int(
            subset["transcription"].str.len().max()
        ),
        "duration_sec": float(subset["duration_sec"].iloc[0])
    })

qc = pd.DataFrame(qc_rows)

assert len(qc) == 100
assert int(qc["empty_transcriptions"].sum()) == 0
assert (qc["duration_sec"] > 0).all()

summary = {
    "batch": "0101-0200",
    "status": "PASS",
    "audio_count": 100,
    "hypothesis_count": 500,
    "raw_variable_audio_count": int(
        (qc["raw_unique"] > 1).sum()
    ),
    "lexically_variable_audio_count": int(
        (qc["lexical_unique"] > 1).sum()
    ),
    "mean_raw_unique": float(qc["raw_unique"].mean()),
    "mean_lexical_unique": float(qc["lexical_unique"].mean()),
    "empty_transcriptions": int(
        qc["empty_transcriptions"].sum()
    ),
    "config_sha256": sha256(CONFIG),
    "asr_csv_sha256": sha256(ASR2),
    "created_at": datetime.now(timezone.utc).isoformat()
}

QC2.parent.mkdir(parents=True, exist_ok=True)

tmp_table = TABLE2.with_suffix(".tmp")
qc.to_csv(tmp_table, index=False, encoding="utf-8")
tmp_table.replace(TABLE2)

QC2.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)

progress["step3_full_batch_0101-0200"] = "qc_pass"
progress["step3_full_batch_0101-0200_qc_report"] = str(QC2)
progress["next_task"] = "Create Batch 2 verified ZIP backup"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 3.3 BATCH 2 QC SUMMARY ===")
for key, value in summary.items():
    print(f"{key}: {value}")

print("\n=== LEXICAL VARIABILITY DISTRIBUTION ===")
print(
    qc["lexical_unique"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nQC report:", QC2)
print("QC table:", TABLE2)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
import zipfile
from pathlib import Path
from datetime import datetime, timezone

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
BACKUP = Path("/kaggle/working/asr_fluency_step03_batch_0101-0200.zip")
CHECK = PROJECT / "checkpoints/research_progress.json"

CONFIG_SHA = "1204e1414ec1332ee2f28caf076a799662d28828dadb8e361243140c388cfc77"
BATCH1_SHA = "b33b7d0e2564c43738a5f254221e639d53b72edf6f5ced0b407962c152e02fb3"
BATCH2_SHA = "e838bb9b722c73aeef2cf10dee6e18ea0970e6df29f4c0e5bcf37b142f52595f"

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

required = [
    "configs/asr_hypothesis_pilot_v1.json",
    "data/metadata/pilot_selection_400.csv",
    "data/metadata/asr_pilot_selection_10.csv",
    "data/metadata/asr_critical_listening_009600118.csv",
    "features/asr/full/asr_hypotheses_0001-0100.csv",
    "features/asr/full/asr_hypotheses_0101-0200.csv",
    "reports/step03_asr_configuration_freeze.json",
    "reports/step03_batch_0001-0100_qc.json",
    "reports/step03_batch_0001-0100_qc_per_audio.csv",
    "reports/step03_batch_0101-0200_qc.json",
    "reports/step03_batch_0101-0200_qc_per_audio.csv",
    "checkpoints/research_progress.json",
]

for rel in required:
    assert (PROJECT / rel).is_file(), f"Missing: {rel}"

assert sha256_file(PROJECT / required[0]) == CONFIG_SHA
assert sha256_file(PROJECT / required[4]) == BATCH1_SHA
assert sha256_file(PROJECT / required[5]) == BATCH2_SHA

qc1 = json.loads(
    (PROJECT / "reports/step03_batch_0001-0100_qc.json")
    .read_text(encoding="utf-8")
)
qc2 = json.loads(
    (PROJECT / "reports/step03_batch_0101-0200_qc.json")
    .read_text(encoding="utf-8")
)

assert qc1["status"] == "PASS"
assert qc2["status"] == "PASS"
assert qc1["asr_csv_sha256"] == BATCH1_SHA
assert qc2["asr_csv_sha256"] == BATCH2_SHA
assert qc2["audio_count"] == 100
assert qc2["hypothesis_count"] == 500

part_dir = PROJECT / "features/asr/full/batch_0101-0200_parts"
parts = sorted(part_dir.glob("asr_*_H1-H5.csv"))

assert len(parts) == 100, (
    f"Expected 100 Batch 2 checkpoints, found {len(parts)}"
)

# Mark the verified QC state before packaging.
progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_full_batch_0001-0100"] = "qc_pass"
progress["step3_full_batch_0101-0200"] = "qc_pass"
progress["step3_batch2_backup_filename"] = BACKUP.name
progress["step3_batch2_backup_status"] = "packaging"
progress["next_task"] = (
    "Verify Batch 2 backup and generate ASR batch 0201-0300"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

files = [PROJECT / rel for rel in required]
files.extend(parts)

# Preserve other small research files if available.
optional_dirs = [
    PROJECT / "features/asr/pilot",
]

for directory in optional_dirs:
    if directory.exists():
        files.extend(p for p in directory.rglob("*") if p.is_file())

optional_files = [
    "reports/step03_pilot_variability_summary.json",
    "reports/step03_pilot_variability_per_audio.csv",
    "reports/step03_pilot_hypothesis_content_audit.csv",
    "reports/step03_critical_asr_review_009600118.md",
]

for rel in optional_files:
    path = PROJECT / rel
    if path.is_file():
        files.append(path)

files = sorted(set(files))

manifest = {
    "archive": BACKUP.name,
    "batch": "0101-0200",
    "batch1_included": True,
    "batch2_qc": "PASS",
    "completed_audio_count": 200,
    "completed_hypothesis_count": 1000,
    "raw_parquet_included": False,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "files": {}
}

tmp = BACKUP.with_suffix(".zip.tmp")

with zipfile.ZipFile(
    tmp,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as archive:

    for path in files:
        relative = path.relative_to(PROJECT).as_posix()
        name = f"acoustic_linguistic_research/{relative}"

        manifest["files"][name] = sha256_file(path)
        archive.write(path, arcname=name)

    archive.writestr(
        "acoustic_linguistic_research/"
        "backup_manifest_batch_0101-0200.json",
        json.dumps(manifest, indent=2, ensure_ascii=False)
    )

tmp.replace(BACKUP)

with zipfile.ZipFile(BACKUP, "r") as archive:
    assert archive.testzip() is None

    saved_manifest = json.loads(
        archive.read(
            "acoustic_linguistic_research/"
            "backup_manifest_batch_0101-0200.json"
        )
    )

    assert saved_manifest["files"] == manifest["files"]

    for name, expected_sha in manifest["files"].items():
        assert sha256_bytes(archive.read(name)) == expected_sha, (
            f"Archive integrity failure: {name}"
        )

print("=== STEP 3.3 BATCH 2 BACKUP VERIFIED ===")
print("Backup:", BACKUP)
print("Files archived:", len(files))
print("Batch 2 audio checkpoints:", len(parts))
print("Total completed audios:", 200)
print("Total completed hypotheses:", 1000)
print("ZIP size MB:", round(BACKUP.stat().st_size / 1024**2, 3))
print("ZIP SHA256:", sha256_file(BACKUP))
print("Batch 1 QC:", qc1["status"])
print("Batch 2 QC:", qc2["status"])
print("Archive integrity: PASS")
print("Next batch: 0201-0300")


In [ ]:

import io
import json
import time
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import torch

from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor
)

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT_DIR = PROJECT / "features/asr/full"
PART_DIR = OUT_DIR / "batch_0201-0300_parts"
FINAL = OUT_DIR / "asr_hypotheses_0201-0300.csv"

PART_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_SHA = (
    "1204e1414ec1332ee2f28caf076a799662d28828dadb8e361243140c388cfc77"
)

PREVIOUS_BATCHES = {
    "0001-0100": "b33b7d0e2564c43738a5f254221e639d53b72edf6f5ced0b407962c152e02fb3",
    "0101-0200": "e838bb9b722c73aeef2cf10dee6e18ea0970e6df29f4c0e5bcf37b142f52595f"
}

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(CONFIG) == CONFIG_SHA, (
    "Frozen configuration SHA256 mismatch"
)

config = json.loads(CONFIG.read_text(encoding="utf-8"))

assert config["status"] == "frozen"
assert config["model_id"] == "openai/whisper-small.en"
assert config["model_revision"] == (
    "e8727524f962ee844a7319d92be39ac1bd25655a"
)

# Verify both previous batches before continuing.
for label, expected_hash in PREVIOUS_BATCHES.items():

    previous_csv = OUT_DIR / f"asr_hypotheses_{label}.csv"
    previous_qc = PROJECT / f"reports/step03_batch_{label}_qc.json"

    assert previous_csv.exists(), f"Missing: {previous_csv}"
    assert previous_qc.exists(), f"Missing: {previous_qc}"

    qc = json.loads(previous_qc.read_text(encoding="utf-8"))

    assert qc["status"] == "PASS"
    assert sha256(previous_csv) == expected_hash
    assert qc["asr_csv_sha256"] == expected_hash
    assert qc["config_sha256"] == CONFIG_SHA

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

batch = pilot.loc[
    pilot["pilot_index"].between(201, 300)
].sort_values("pilot_index").copy()

assert len(batch) == 100
assert batch["pilot_index"].tolist() == list(range(201, 301))
assert batch["audio_id"].is_unique
assert batch["speaker"].notna().all()

hypotheses = config["hypotheses"]

assert [h["id"] for h in hypotheses] == [
    "H1", "H2", "H3", "H4", "H5"
]

specs = {h["id"]: h for h in hypotheses}

expected_columns = [
    "pilot_index", "audio_id", "speaker",
    "hypothesis_id", "method",
    "model_id", "model_revision",
    "do_sample", "num_beams", "temperature",
    "seed", "transcription", "duration_sec",
    "inference_sec"
]

def validate_part(path, row):
    df = pd.read_csv(
        path,
        dtype={
            "audio_id": "string",
            "speaker": "string",
            "seed": "string"
        },
        keep_default_na=False
    )

    assert len(df) == 5
    assert df.columns.tolist() == expected_columns
    assert df["pilot_index"].eq(int(row["pilot_index"])).all()
    assert df["audio_id"].eq(str(row["audio_id"])).all()
    assert df["speaker"].eq(str(row["speaker"])).all()
    assert df["hypothesis_id"].tolist() == [
        "H1", "H2", "H3", "H4", "H5"
    ]
    assert df["model_id"].eq(config["model_id"]).all()
    assert df["model_revision"].eq(
        config["model_revision"]
    ).all()

    assert not df.duplicated(
        ["audio_id", "hypothesis_id"]
    ).any()

    for _, record in df.iterrows():
        spec = specs[record["hypothesis_id"]]

        assert record["method"] == spec["method"]

        assert isinstance(record["transcription"], str)
        assert record["transcription"].strip() != ""

        if spec["method"] == "sampling":
            assert str(record["do_sample"]).lower() == "true"
            assert int(record["num_beams"]) == 1

            assert np.isclose(
                float(record["temperature"]),
                float(spec["temperature"])
            )

            # Accept CSV representations "101" and "101.0".
            assert float(record["seed"]) == float(spec["seed"])

        else:
            assert str(record["do_sample"]).lower() == "false"

            expected_beams = (
                int(spec["num_beams"])
                if spec["method"] == "beam_search"
                else 1
            )

            assert int(record["num_beams"]) == expected_beams
            assert str(record["seed"]) == ""
            assert str(record["temperature"]) == ""

        assert np.isfinite(float(record["duration_sec"]))
        assert float(record["duration_sec"]) > 0
        assert np.isfinite(float(record["inference_sec"]))
        assert float(record["inference_sec"]) >= 0

    return df

# Identify completed work before loading Whisper.
pending = []
completed = 0

for _, row in batch.iterrows():
    part = PART_DIR / f"asr_{row['audio_id']}_H1-H5.csv"

    if part.exists():
        validate_part(part, row)
        completed += 1
    else:
        pending.append(row)

print("=== STEP 3.3: ASR BATCH 0201-0300 ===")
print("Model:", config["model_id"])
print("Revision:", config["model_revision"])
print("Frozen config SHA256:", sha256(CONFIG))
print("Total audios:", len(batch))
print("Completed checkpoints:", completed)
print("Pending audios:", len(pending))
print("Expected hypotheses:", 500)
print("Previous batches: QC PASS")

if pending:

    assert TRAIN.exists(), "Train Parquet missing"
    assert torch.cuda.is_available(), "CUDA GPU unavailable"

    wanted = {str(row["audio_id"]) for row in pending}
    audio_data = {}

    pf = pq.ParquetFile(TRAIN)

    assert "audio" in pf.schema_arrow.names

    for group_id in range(pf.metadata.num_row_groups):

        table = pf.read_row_group(
            group_id,
            columns=["audio"]
        )

        for audio in table.column("audio").to_pylist():

            audio_id = Path(audio["path"]).stem

            if audio_id in wanted:
                assert audio_id not in audio_data
                audio_data[audio_id] = audio["bytes"]

    assert set(audio_data) == wanted, (
        "Some selected audio files were not found"
    )

    device = "cuda:0"

    processor = WhisperProcessor.from_pretrained(
        config["model_id"],
        revision=config["model_revision"]
    )

    model = WhisperForConditionalGeneration.from_pretrained(
        config["model_id"],
        revision=config["model_revision"],
        dtype=torch.float16
    ).to(device)

    model.eval()

    for count, row in enumerate(pending, start=1):

        audio_id = str(row["audio_id"])
        pilot_index = int(row["pilot_index"])

        y, sr = sf.read(
            io.BytesIO(audio_data[audio_id]),
            dtype="float32"
        )

        assert sr == 16000
        assert y.ndim == 1
        assert len(y) > 0
        assert np.isfinite(y).all()

        inputs = processor(
            y,
            sampling_rate=sr,
            return_tensors="pt"
        )

        input_features = inputs.input_features.to(
            device=device,
            dtype=torch.float16
        )

        records = []

        for h in hypotheses:

            kwargs = {
                "do_sample": False,
                "num_beams": 1,
                "max_new_tokens": 128
            }

            seed = None
            temperature = None

            if h["method"] == "beam_search":

                kwargs["num_beams"] = int(h["num_beams"])

            elif h["method"] == "sampling":

                seed = int(h["seed"])
                temperature = float(h["temperature"])

                kwargs["do_sample"] = True
                kwargs["temperature"] = temperature

                torch.manual_seed(seed)
                torch.cuda.manual_seed_all(seed)

            torch.cuda.synchronize()
            start = time.perf_counter()

            with torch.inference_mode():

                generated_ids = model.generate(
                    input_features,
                    **kwargs
                )

            torch.cuda.synchronize()
            elapsed = time.perf_counter() - start

            transcription = processor.batch_decode(
                generated_ids,
                skip_special_tokens=True
            )[0].strip()

            records.append({
                "pilot_index": pilot_index,
                "audio_id": audio_id,
                "speaker": str(row["speaker"]),
                "hypothesis_id": h["id"],
                "method": h["method"],
                "model_id": config["model_id"],
                "model_revision": config["model_revision"],
                "do_sample": kwargs["do_sample"],
                "num_beams": kwargs["num_beams"],
                "temperature": temperature,
                "seed": seed,
                "transcription": transcription,
                "duration_sec": len(y) / sr,
                "inference_sec": elapsed
            })

        result = pd.DataFrame(
            records,
            columns=expected_columns
        )

        part = PART_DIR / f"asr_{audio_id}_H1-H5.csv"
        temp = part.with_suffix(".tmp")

        result.to_csv(
            temp,
            index=False,
            encoding="utf-8"
        )
        temp.replace(part)

        validate_part(part, row)

        if (
            count == 1
            or count % 10 == 0
            or count == len(pending)
        ):
            print(
                f"[{count:03d}/{len(pending):03d}] "
                f"COMPLETE index={pilot_index} "
                f"audio={audio_id}"
            )

# Assemble the complete Batch 3 in pilot-index order.
pieces = []

for _, row in batch.iterrows():

    audio_id = str(row["audio_id"])

    part = PART_DIR / f"asr_{audio_id}_H1-H5.csv"

    pieces.append(validate_part(part, row))

combined = pd.concat(pieces, ignore_index=True)

assert len(combined) == 500
assert combined["audio_id"].nunique() == 100
assert combined["pilot_index"].nunique() == 100

assert not combined.duplicated(
    ["audio_id", "hypothesis_id"]
).any()

temp_final = FINAL.with_suffix(".tmp")

combined.to_csv(
    temp_final,
    index=False,
    encoding="utf-8"
)
temp_final.replace(FINAL)

final_sha = sha256(FINAL)

unique_counts = combined.groupby(
    "audio_id"
)["transcription"].nunique()

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)

progress["step3_full_batch_0201-0300"] = "completed_pending_qc"
progress["step3_full_batch_0201-0300_sha256"] = final_sha
progress["step3_full_generated_audio_count"] = 300
progress["step3_full_generated_hypothesis_count"] = 1500
progress["next_task"] = (
    "Quality control for ASR batch 0201-0300"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(
        progress,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("\n=== ASR BATCH 0201-0300 SUMMARY ===")
print("Audios:", combined["audio_id"].nunique())
print("Hypotheses:", len(combined))
print("Mean unique transcripts:", round(unique_counts.mean(), 3))
print(
    "Audios with raw variability:",
    int((unique_counts > 1).sum())
)
print(
    "Empty transcriptions:",
    int(combined["transcription"].str.strip().eq("").sum())
)
print("CSV:", FINAL)
print("SHA256:", final_sha)
print("Checkpoint updated.")


In [ ]:

import json
import re
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
ASR_DIR = PROJECT / "features/asr/full"
REPORT_DIR = PROJECT / "reports"
CHECK = PROJECT / "checkpoints/research_progress.json"

ASR3 = ASR_DIR / "asr_hypotheses_0201-0300.csv"
QC3 = REPORT_DIR / "step03_batch_0201-0300_qc.json"
TABLE3 = REPORT_DIR / "step03_batch_0201-0300_qc_per_audio.csv"

CONFIG_SHA = "1204e1414ec1332ee2f28caf076a799662d28828dadb8e361243140c388cfc77"
ASR3_SHA = "bde72d422c46b9eb357cfc6f7bac702a6a416b302329a1e3582afdae54bfee51"

PREVIOUS = {
    "0001-0100": "b33b7d0e2564c43738a5f254221e639d53b72edf6f5ced0b407962c152e02fb3",
    "0101-0200": "e838bb9b722c73aeef2cf10dee6e18ea0970e6df29f4c0e5bcf37b142f52595f"
}

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def normalize_text(value):
    value = str(value).lower()
    value = re.sub(r"[^\w\s']", " ", value)
    return re.sub(r"\s+", " ", value).strip()

assert sha256(CONFIG) == CONFIG_SHA
assert sha256(ASR3) == ASR3_SHA

config = json.loads(CONFIG.read_text(encoding="utf-8"))
assert config["status"] == "frozen"

# Verify earlier batches are unchanged.
for label, expected_sha in PREVIOUS.items():
    csv_path = ASR_DIR / f"asr_hypotheses_{label}.csv"
    qc_path = REPORT_DIR / f"step03_batch_{label}_qc.json"

    previous_qc = json.loads(
        qc_path.read_text(encoding="utf-8")
    )

    assert previous_qc["status"] == "PASS"
    assert sha256(csv_path) == expected_sha
    assert previous_qc["asr_csv_sha256"] == expected_sha

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

expected = pilot.loc[
    pilot["pilot_index"].between(201, 300),
    ["pilot_index", "audio_id", "speaker"]
].sort_values("pilot_index")

df = pd.read_csv(
    ASR3,
    dtype={
        "audio_id": "string",
        "speaker": "string",
        "seed": "string"
    },
    keep_default_na=False
)

assert len(expected) == 100
assert len(df) == 500
assert df["audio_id"].nunique() == 100
assert df["pilot_index"].nunique() == 100
assert not df.duplicated(
    ["audio_id", "hypothesis_id"]
).any()

assert df["model_id"].eq(config["model_id"]).all()
assert df["model_revision"].eq(config["model_revision"]).all()

hypothesis_ids = ["H1", "H2", "H3", "H4", "H5"]
specs = {h["id"]: h for h in config["hypotheses"]}
assert set(specs) == set(hypothesis_ids)

qc_rows = []

for _, item in expected.iterrows():
    audio_id = str(item["audio_id"])

    subset = df.loc[
        df["audio_id"] == audio_id
    ].sort_values("hypothesis_id")

    assert len(subset) == 5
    assert subset["hypothesis_id"].tolist() == hypothesis_ids
    assert subset["pilot_index"].eq(
        int(item["pilot_index"])
    ).all()
    assert subset["speaker"].eq(
        str(item["speaker"])
    ).all()

    for _, record in subset.iterrows():
        spec = specs[record["hypothesis_id"]]

        assert record["method"] == spec["method"]
        assert str(record["transcription"]).strip() != ""

        if spec["method"] == "sampling":
            assert str(record["do_sample"]).lower() == "true"
            assert int(record["num_beams"]) == 1
            assert np.isclose(
                float(record["temperature"]),
                float(spec["temperature"])
            )
            assert float(record["seed"]) == float(spec["seed"])
        else:
            assert str(record["do_sample"]).lower() == "false"

            expected_beams = (
                int(spec["num_beams"])
                if spec["method"] == "beam_search"
                else 1
            )

            assert int(record["num_beams"]) == expected_beams
            assert str(record["temperature"]) == ""
            assert str(record["seed"]) == ""

        assert np.isfinite(float(record["duration_sec"]))
        assert float(record["duration_sec"]) > 0
        assert np.isfinite(float(record["inference_sec"]))
        assert float(record["inference_sec"]) >= 0

    normalized = subset["transcription"].map(normalize_text)

    qc_rows.append({
        "pilot_index": int(item["pilot_index"]),
        "audio_id": audio_id,
        "speaker": str(item["speaker"]),
        "raw_unique": int(subset["transcription"].nunique()),
        "lexical_unique": int(normalized.nunique()),
        "empty_transcriptions": int(
            subset["transcription"].str.strip().eq("").sum()
        ),
        "max_transcription_chars": int(
            subset["transcription"].str.len().max()
        ),
        "duration_sec": float(subset["duration_sec"].iloc[0])
    })

qc = pd.DataFrame(qc_rows)

assert len(qc) == 100
assert int(qc["empty_transcriptions"].sum()) == 0
assert (qc["duration_sec"] > 0).all()
assert (qc["lexical_unique"] <= qc["raw_unique"]).all()

summary = {
    "batch": "0201-0300",
    "status": "PASS",
    "audio_count": 100,
    "hypothesis_count": 500,
    "raw_variable_audio_count": int(
        (qc["raw_unique"] > 1).sum()
    ),
    "lexically_variable_audio_count": int(
        (qc["lexical_unique"] > 1).sum()
    ),
    "mean_raw_unique": float(qc["raw_unique"].mean()),
    "mean_lexical_unique": float(qc["lexical_unique"].mean()),
    "empty_transcriptions": int(
        qc["empty_transcriptions"].sum()
    ),
    "config_sha256": sha256(CONFIG),
    "asr_csv_sha256": sha256(ASR3),
    "created_at": datetime.now(timezone.utc).isoformat()
}

# Verify summary agrees with generation output.
assert summary["raw_variable_audio_count"] == 38
assert np.isclose(summary["mean_raw_unique"], 1.62)

REPORT_DIR.mkdir(parents=True, exist_ok=True)

tmp_table = TABLE3.with_suffix(".tmp")
qc.to_csv(tmp_table, index=False, encoding="utf-8")
tmp_table.replace(TABLE3)

tmp_report = QC3.with_suffix(".tmp")
tmp_report.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)
tmp_report.replace(QC3)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_full_batch_0201-0300"] = "qc_pass"
progress["step3_full_batch_0201-0300_qc_report"] = str(QC3)
progress["next_task"] = "Create Batch 3 verified ZIP backup"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 3.3 BATCH 3 QC SUMMARY ===")
for key, value in summary.items():
    print(f"{key}: {value}")

print("\n=== LEXICAL VARIABILITY DISTRIBUTION ===")
print(
    qc["lexical_unique"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nQC report:", QC3)
print("QC table:", TABLE3)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
import zipfile
from pathlib import Path
from datetime import datetime, timezone

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
BACKUP = Path("/kaggle/working/asr_fluency_step03_batch_0201-0300.zip")
CHECK = PROJECT / "checkpoints/research_progress.json"

CONFIG_SHA = "1204e1414ec1332ee2f28caf076a799662d28828dadb8e361243140c388cfc77"

BATCH_SHAS = {
    "0001-0100": "b33b7d0e2564c43738a5f254221e639d53b72edf6f5ced0b407962c152e02fb3",
    "0101-0200": "e838bb9b722c73aeef2cf10dee6e18ea0970e6df29f4c0e5bcf37b142f52595f",
    "0201-0300": "bde72d422c46b9eb357cfc6f7bac702a6a416b302329a1e3582afdae54bfee51"
}

def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
assert sha256_file(CONFIG) == CONFIG_SHA

required = [
    "configs/asr_hypothesis_pilot_v1.json",
    "data/metadata/pilot_selection_400.csv",
    "data/metadata/asr_pilot_selection_10.csv",
    "data/metadata/asr_critical_listening_009600118.csv",
    "reports/step03_asr_configuration_freeze.json",
    "checkpoints/research_progress.json",
]

for label, expected_sha in BATCH_SHAS.items():
    csv_relative = f"features/asr/full/asr_hypotheses_{label}.csv"
    qc_relative = f"reports/step03_batch_{label}_qc.json"
    table_relative = f"reports/step03_batch_{label}_qc_per_audio.csv"

    required.extend([csv_relative, qc_relative, table_relative])

    csv_path = PROJECT / csv_relative
    qc_path = PROJECT / qc_relative

    assert csv_path.is_file()
    assert qc_path.is_file()
    assert sha256_file(csv_path) == expected_sha

    qc = json.loads(qc_path.read_text(encoding="utf-8"))

    assert qc["status"] == "PASS"
    assert qc["audio_count"] == 100
    assert qc["hypothesis_count"] == 500
    assert qc["config_sha256"] == CONFIG_SHA
    assert qc["asr_csv_sha256"] == expected_sha

for relative in required:
    assert (PROJECT / relative).is_file(), (
        f"Missing required file: {relative}"
    )

# Verify Batch 3 per-audio checkpoints.
PART_DIR = PROJECT / "features/asr/full/batch_0201-0300_parts"
parts = sorted(PART_DIR.glob("asr_*_H1-H5.csv"))

assert len(parts) == 100, (
    f"Expected 100 checkpoints, found {len(parts)}"
)

# Update the checkpoint before packaging.
progress = json.loads(CHECK.read_text(encoding="utf-8"))

for label in BATCH_SHAS:
    assert progress.get(
        f"step3_full_batch_{label}"
    ) == "qc_pass", f"QC state mismatch: {label}"

progress["step3_batch3_backup_filename"] = BACKUP.name
progress["step3_batch3_backup_status"] = "packaging"
progress["step3_full_generated_audio_count"] = 300
progress["step3_full_generated_hypothesis_count"] = 1500
progress["next_task"] = (
    "Verify Batch 3 backup and start ASR Batch 0301-0400"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

files = [PROJECT / relative for relative in required]
files.extend(parts)

# Include existing pilot reports and small ASR artifacts.
optional_files = [
    "reports/step03_pilot_variability_summary.json",
    "reports/step03_pilot_variability_per_audio.csv",
    "reports/step03_pilot_hypothesis_content_audit.csv",
    "reports/step03_critical_asr_review_009600118.md",
    "reports/step03_whisper_smoke_test.json",
    "reports/step03_whisper_beam_test.json",
    "reports/step03_sampling_smoke_test.json"
]

for relative in optional_files:
    path = PROJECT / relative
    if path.is_file():
        files.append(path)

for directory in [
    PROJECT / "features/asr/pilot"
]:
    if directory.exists():
        files.extend(
            p for p in directory.rglob("*") if p.is_file()
        )

files.extend(
    PROJECT.glob("features/asr/asr_smoke_test_*.csv")
)

files = sorted(set(files))

manifest = {
    "backup_name": BACKUP.name,
    "batch": "0201-0300",
    "batch_csv_ranges_included": list(BATCH_SHAS.keys()),
    "batch3_individual_checkpoints": 100,
    "completed_audio_count": 300,
    "completed_hypothesis_count": 1500,
    "qc_status": "PASS",
    "raw_parquet_included": False,
    "whisper_weights_included": False,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "files": {}
}

temp_zip = BACKUP.with_suffix(".zip.tmp")

with zipfile.ZipFile(
    temp_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as archive:

    for path in files:
        relative = path.relative_to(PROJECT).as_posix()
        archive_name = (
            "acoustic_linguistic_research/" + relative
        )

        manifest["files"][archive_name] = sha256_file(path)
        archive.write(path, arcname=archive_name)

    manifest_name = (
        "acoustic_linguistic_research/"
        "backup_manifest_batch_0201-0300.json"
    )

    archive.writestr(
        manifest_name,
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False
        )
    )

temp_zip.replace(BACKUP)

# Verify all archived bytes against the manifest.
with zipfile.ZipFile(BACKUP, "r") as archive:

    assert archive.testzip() is None

    saved_manifest = json.loads(
        archive.read(manifest_name)
    )

    assert saved_manifest["files"] == manifest["files"]

    for name, expected_hash in manifest["files"].items():
        actual_hash = sha256_bytes(archive.read(name))
        assert actual_hash == expected_hash, (
            f"Archive integrity error: {name}"
        )

# Mark backup as verified in the live checkpoint.
# The archived checkpoint intentionally records "packaging".
progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_batch3_backup_status"] = "verified"
progress["step3_batch3_backup_sha256"] = sha256_file(BACKUP)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 3.3 BATCH 3 BACKUP VERIFIED ===")
print("Backup:", BACKUP)
print("Files archived:", len(files))
print("Batch 3 audio checkpoints:", len(parts))
print("Completed audios:", 300)
print("Completed hypotheses:", 1500)
print("ZIP size MB:", round(BACKUP.stat().st_size / 1024**2, 3))
print("ZIP SHA256:", sha256_file(BACKUP))
print("Batch 1 QC: PASS")
print("Batch 2 QC: PASS")
print("Batch 3 QC: PASS")
print("Archive integrity: PASS")
print("Next batch: 0301-0400")
print("Checkpoint updated.")


In [ ]:

import io
import json
import time
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import torch

from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor
)

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

OUT_DIR = PROJECT / "features/asr/full"
PART_DIR = OUT_DIR / "batch_0301-0400_parts"
FINAL = OUT_DIR / "asr_hypotheses_0301-0400.csv"

PART_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_SHA = (
    "1204e1414ec1332ee2f28caf076a799662d28828dadb8e361243140c388cfc77"
)

PREVIOUS_BATCHES = {
    "0001-0100": (
        "b33b7d0e2564c43738a5f254221e639d53b72edf6f5ced0b407962c152e02fb3"
    ),
    "0101-0200": (
        "e838bb9b722c73aeef2cf10dee6e18ea0970e6df29f4c0e5bcf37b142f52595f"
    ),
    "0201-0300": (
        "bde72d422c46b9eb357cfc6f7bac702a6a416b302329a1e3582afdae54bfee51"
    )
}

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(CONFIG) == CONFIG_SHA, (
    "Frozen configuration SHA256 mismatch"
)

config = json.loads(CONFIG.read_text(encoding="utf-8"))

assert config["status"] == "frozen"
assert config["model_id"] == "openai/whisper-small.en"
assert config["model_revision"] == (
    "e8727524f962ee844a7319d92be39ac1bd25655a"
)

# Verify all previous batches.
for label, expected_hash in PREVIOUS_BATCHES.items():

    previous_csv = (
        OUT_DIR / f"asr_hypotheses_{label}.csv"
    )
    previous_qc = (
        PROJECT / f"reports/step03_batch_{label}_qc.json"
    )

    assert previous_csv.is_file()
    assert previous_qc.is_file()

    qc = json.loads(
        previous_qc.read_text(encoding="utf-8")
    )

    assert qc["status"] == "PASS"
    assert qc["audio_count"] == 100
    assert qc["hypothesis_count"] == 500
    assert qc["config_sha256"] == CONFIG_SHA
    assert qc["asr_csv_sha256"] == expected_hash
    assert sha256(previous_csv) == expected_hash

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

batch = pilot.loc[
    pilot["pilot_index"].between(301, 400)
].sort_values("pilot_index").copy()

assert len(batch) == 100
assert batch["pilot_index"].tolist() == list(range(301, 401))
assert batch["audio_id"].is_unique
assert batch["speaker"].notna().all()

hypotheses = config["hypotheses"]

assert len(hypotheses) == 5
assert [h["id"] for h in hypotheses] == [
    "H1", "H2", "H3", "H4", "H5"
]

specs = {h["id"]: h for h in hypotheses}

assert [specs[h]["method"] for h in ["H1", "H2"]] == [
    "greedy", "beam_search"
]

for hid, seed in [("H3", 101), ("H4", 202), ("H5", 303)]:
    assert specs[hid]["method"] == "sampling"
    assert int(specs[hid]["seed"]) == seed
    assert float(specs[hid]["temperature"]) == 0.3

expected_columns = [
    "pilot_index", "audio_id", "speaker",
    "hypothesis_id", "method",
    "model_id", "model_revision",
    "do_sample", "num_beams", "temperature",
    "seed", "transcription", "duration_sec",
    "inference_sec"
]

def validate_part(path, row):

    df = pd.read_csv(
        path,
        dtype={
            "audio_id": "string",
            "speaker": "string",
            "seed": "string"
        },
        keep_default_na=False
    )

    assert df.columns.tolist() == expected_columns
    assert len(df) == 5
    assert df["pilot_index"].eq(
        int(row["pilot_index"])
    ).all()
    assert df["audio_id"].eq(
        str(row["audio_id"])
    ).all()
    assert df["speaker"].eq(
        str(row["speaker"])
    ).all()

    assert df["hypothesis_id"].tolist() == [
        "H1", "H2", "H3", "H4", "H5"
    ]

    assert df["model_id"].eq(
        config["model_id"]
    ).all()

    assert df["model_revision"].eq(
        config["model_revision"]
    ).all()

    assert not df.duplicated(
        ["audio_id", "hypothesis_id"]
    ).any()

    for _, record in df.iterrows():

        spec = specs[record["hypothesis_id"]]

        assert record["method"] == spec["method"]
        assert isinstance(record["transcription"], str)
        assert record["transcription"].strip() != ""

        if spec["method"] == "sampling":

            assert str(
                record["do_sample"]
            ).lower() == "true"

            assert int(record["num_beams"]) == 1

            assert np.isclose(
                float(record["temperature"]),
                float(spec["temperature"])
            )

            # Accept both "101" and "101.0".
            assert float(record["seed"]) == float(
                spec["seed"]
            )

        else:

            assert str(
                record["do_sample"]
            ).lower() == "false"

            expected_beams = (
                int(spec["num_beams"])
                if spec["method"] == "beam_search"
                else 1
            )

            assert int(
                record["num_beams"]
            ) == expected_beams

            assert str(record["temperature"]) == ""
            assert str(record["seed"]) == ""

        duration = float(record["duration_sec"])
        elapsed = float(record["inference_sec"])

        assert np.isfinite(duration) and duration > 0
        assert np.isfinite(elapsed) and elapsed >= 0

    return df


# Identify finished checkpoints before loading the model.
pending = []
completed = 0

for _, row in batch.iterrows():

    audio_id = str(row["audio_id"])
    part = PART_DIR / f"asr_{audio_id}_H1-H5.csv"

    if part.exists():
        validate_part(part, row)
        completed += 1
    else:
        pending.append(row)

print("=== STEP 3.3: ASR BATCH 0301-0400 ===")
print("Model:", config["model_id"])
print("Revision:", config["model_revision"])
print("Config SHA256:", sha256(CONFIG))
print("Previous batches: QC PASS")
print("Total audios:", len(batch))
print("Completed checkpoints:", completed)
print("Pending audios:", len(pending))
print("Expected hypotheses:", 500)

if pending:

    assert TRAIN.exists(), "Train Parquet missing"
    assert torch.cuda.is_available(), "CUDA GPU unavailable"

    wanted = {
        str(row["audio_id"])
        for row in pending
    }

    audio_data = {}
    pf = pq.ParquetFile(TRAIN)

    # Use schema_arrow.names, NOT schema.names.
    assert "audio" in pf.schema_arrow.names

    for group_id in range(pf.metadata.num_row_groups):

        table = pf.read_row_group(
            group_id,
            columns=["audio"]
        )

        for audio in table.column("audio").to_pylist():

            audio_id = Path(audio["path"]).stem

            if audio_id in wanted:

                assert audio_id not in audio_data
                audio_data[audio_id] = audio["bytes"]

    assert set(audio_data) == wanted, (
        "Some selected audio recordings are missing."
    )

    device = "cuda:0"

    processor = WhisperProcessor.from_pretrained(
        config["model_id"],
        revision=config["model_revision"]
    )

    model = WhisperForConditionalGeneration.from_pretrained(
        config["model_id"],
        revision=config["model_revision"],
        dtype=torch.float16
    ).to(device)

    model.eval()

    for count, row in enumerate(pending, start=1):

        audio_id = str(row["audio_id"])
        pilot_index = int(row["pilot_index"])

        y, sr = sf.read(
            io.BytesIO(audio_data[audio_id]),
            dtype="float32"
        )

        assert sr == 16000
        assert y.ndim == 1
        assert len(y) > 0
        assert np.isfinite(y).all()

        inputs = processor(
            y,
            sampling_rate=sr,
            return_tensors="pt"
        )

        input_features = inputs.input_features.to(
            device=device,
            dtype=torch.float16
        )

        records = []

        for h in hypotheses:

            kwargs = {
                "do_sample": False,
                "num_beams": 1,
                "max_new_tokens": 128
            }

            seed = None
            temperature = None

            if h["method"] == "beam_search":

                kwargs["num_beams"] = int(
                    h["num_beams"]
                )

            elif h["method"] == "sampling":

                seed = int(h["seed"])
                temperature = float(h["temperature"])

                kwargs["do_sample"] = True
                kwargs["temperature"] = temperature

                torch.manual_seed(seed)
                torch.cuda.manual_seed_all(seed)

            torch.cuda.synchronize()
            start = time.perf_counter()

            with torch.inference_mode():

                generated_ids = model.generate(
                    input_features,
                    **kwargs
                )

            torch.cuda.synchronize()
            elapsed = time.perf_counter() - start

            transcription = processor.batch_decode(
                generated_ids,
                skip_special_tokens=True
            )[0].strip()

            records.append({
                "pilot_index": pilot_index,
                "audio_id": audio_id,
                "speaker": str(row["speaker"]),
                "hypothesis_id": h["id"],
                "method": h["method"],
                "model_id": config["model_id"],
                "model_revision": config["model_revision"],
                "do_sample": kwargs["do_sample"],
                "num_beams": kwargs["num_beams"],
                "temperature": temperature,
                "seed": seed,
                "transcription": transcription,
                "duration_sec": len(y) / sr,
                "inference_sec": elapsed
            })

        result = pd.DataFrame(
            records,
            columns=expected_columns
        )

        # Save each completed recording atomically.
        part = PART_DIR / f"asr_{audio_id}_H1-H5.csv"
        temp = part.with_suffix(".tmp")

        result.to_csv(
            temp,
            index=False,
            encoding="utf-8"
        )

        temp.replace(part)
        validate_part(part, row)

        if (
            count == 1
            or count % 10 == 0
            or count == len(pending)
        ):
            print(
                f"[{count:03d}/{len(pending):03d}] "
                f"COMPLETE index={pilot_index} "
                f"audio={audio_id}"
            )

# Combine all verified audio checkpoints.
pieces = []

for _, row in batch.iterrows():

    audio_id = str(row["audio_id"])
    part = PART_DIR / f"asr_{audio_id}_H1-H5.csv"

    pieces.append(validate_part(part, row))

combined = pd.concat(pieces, ignore_index=True)

assert len(combined) == 500
assert combined["audio_id"].nunique() == 100
assert combined["pilot_index"].nunique() == 100

assert not combined.duplicated(
    ["audio_id", "hypothesis_id"]
).any()

# Avoid accidentally replacing an existing final
# result with different bytes.
temp_final = FINAL.with_suffix(".tmp")

combined.to_csv(
    temp_final,
    index=False,
    encoding="utf-8"
)

if FINAL.exists():
    assert sha256(FINAL) == sha256(temp_final), (
        "Existing final Batch 4 CSV differs. "
        "Existing file preserved; investigate before overwrite."
    )
    temp_final.unlink()
else:
    temp_final.replace(FINAL)

final_sha = sha256(FINAL)

unique_counts = combined.groupby(
    "audio_id"
)["transcription"].nunique()

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)

progress["step3_full_batch_0301-0400"] = "completed_pending_qc"
progress["step3_full_batch_0301-0400_sha256"] = final_sha
progress["step3_full_generated_audio_count"] = 400
progress["step3_full_generated_hypothesis_count"] = 2000
progress["next_task"] = (
    "Quality control for ASR Batch 0301-0400"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(
        progress,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("\n=== ASR BATCH 0301-0400 SUMMARY ===")
print("Audios:", combined["audio_id"].nunique())
print("Hypotheses:", len(combined))
print(
    "Mean unique transcripts:",
    round(unique_counts.mean(), 3)
)
print(
    "Audios with raw variability:",
    int((unique_counts > 1).sum())
)
print(
    "Empty transcriptions:",
    int(combined["transcription"].str.strip().eq("").sum())
)
print("CSV:", FINAL)
print("SHA256:", final_sha)
print("Checkpoint updated.")


In [ ]:

import json
import re
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
ASR_DIR = PROJECT / "features/asr/full"
REPORT_DIR = PROJECT / "reports"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CONFIG = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
CHECK = PROJECT / "checkpoints/research_progress.json"

CONFIG_SHA = "1204e1414ec1332ee2f28caf076a799662d28828dadb8e361243140c388cfc77"

EXPECTED_SHAS = {
    "0001-0100": "b33b7d0e2564c43738a5f254221e639d53b72edf6f5ced0b407962c152e02fb3",
    "0101-0200": "e838bb9b722c73aeef2cf10dee6e18ea0970e6df29f4c0e5bcf37b142f52595f",
    "0201-0300": "bde72d422c46b9eb357cfc6f7bac702a6a416b302329a1e3582afdae54bfee51",
    "0301-0400": "552a52ee21cbda8a07722f60baafee67114542316d3c602243ef5d0cbae1c777"
}

BATCH4_QC = REPORT_DIR / "step03_batch_0301-0400_qc.json"
BATCH4_TABLE = REPORT_DIR / "step03_batch_0301-0400_qc_per_audio.csv"
FULL_CSV = ASR_DIR / "asr_hypotheses_0001-0400.csv"
FULL_QC = REPORT_DIR / "step03_full_0400_integrity_qc.json"
FULL_TABLE = REPORT_DIR / "step03_full_0400_variability_per_audio.csv"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def normalize_text(value):
    value = str(value).lower()
    value = re.sub(r"[^\w\s']", " ", value)
    return re.sub(r"\s+", " ", value).strip()

def atomic_csv(df, path):
    tmp = path.with_suffix(".tmp")
    df.to_csv(tmp, index=False, encoding="utf-8")
    if path.exists():
        assert sha256(path) == sha256(tmp), (
            f"Existing file differs: {path}"
        )
        tmp.unlink()
    else:
        tmp.replace(path)

def atomic_json(data, path):
    if path.exists():
        existing = json.loads(path.read_text(encoding="utf-8"))
        for key in data:
            if key != "created_at":
                assert existing.get(key) == data[key], (
                    f"Existing report differs: {path}, field={key}"
                )
        return
    tmp = path.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(data, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp.replace(path)

assert sha256(CONFIG) == CONFIG_SHA
config = json.loads(CONFIG.read_text(encoding="utf-8"))
assert config["status"] == "frozen"

specs = {h["id"]: h for h in config["hypotheses"]}
HIDS = ["H1", "H2", "H3", "H4", "H5"]
assert list(specs.keys()) == HIDS

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
).sort_values("pilot_index")

assert len(pilot) == 400
assert pilot["pilot_index"].tolist() == list(range(1, 401))
assert pilot["audio_id"].is_unique

expected_columns = [
    "pilot_index", "audio_id", "speaker",
    "hypothesis_id", "method", "model_id",
    "model_revision", "do_sample", "num_beams",
    "temperature", "seed", "transcription",
    "duration_sec", "inference_sec"
]

pieces = []

for label, expected_hash in EXPECTED_SHAS.items():
    path = ASR_DIR / f"asr_hypotheses_{label}.csv"
    assert sha256(path) == expected_hash, (
        f"Hash mismatch: {label}"
    )

    df = pd.read_csv(
        path,
        dtype={
            "audio_id": "string",
            "speaker": "string",
            "seed": "string"
        },
        keep_default_na=False
    )

    assert df.columns.tolist() == expected_columns
    assert len(df) == 500
    assert df["model_id"].eq(config["model_id"]).all()
    assert df["model_revision"].eq(
        config["model_revision"]
    ).all()

    pieces.append(df)

combined = pd.concat(pieces, ignore_index=True)

assert len(combined) == 2000
assert combined["audio_id"].nunique() == 400
assert combined["pilot_index"].nunique() == 400
assert not combined.duplicated(
    ["pilot_index", "audio_id", "hypothesis_id"]
).any()

qc_rows = []

for _, item in pilot.iterrows():
    idx = int(item["pilot_index"])
    audio_id = str(item["audio_id"])
    speaker = str(item["speaker"])

    subset = combined.loc[
        combined["pilot_index"] == idx
    ].sort_values("hypothesis_id")

    assert len(subset) == 5
    assert subset["hypothesis_id"].tolist() == HIDS
    assert subset["audio_id"].eq(audio_id).all()
    assert subset["speaker"].eq(speaker).all()

    durations = subset["duration_sec"].astype(float)
    assert np.isfinite(durations).all()
    assert (durations > 0).all()
    assert np.allclose(durations, durations.iloc[0])

    for _, record in subset.iterrows():
        spec = specs[record["hypothesis_id"]]

        assert record["method"] == spec["method"]
        assert str(record["transcription"]).strip() != ""

        assert np.isfinite(float(record["inference_sec"]))
        assert float(record["inference_sec"]) >= 0

        if spec["method"] == "sampling":
            assert str(record["do_sample"]).lower() == "true"
            assert int(record["num_beams"]) == 1
            assert np.isclose(
                float(record["temperature"]),
                float(spec["temperature"])
            )
            assert float(record["seed"]) == float(spec["seed"])
        else:
            assert str(record["do_sample"]).lower() == "false"
            beams = (
                int(spec["num_beams"])
                if spec["method"] == "beam_search"
                else 1
            )
            assert int(record["num_beams"]) == beams
            assert str(record["seed"]) == ""
            assert str(record["temperature"]) == ""

    norm = subset["transcription"].map(normalize_text)

    qc_rows.append({
        "pilot_index": idx,
        "audio_id": audio_id,
        "speaker": speaker,
        "batch": f"{((idx - 1) // 100) * 100 + 1:04d}-"
                 f"{((idx - 1) // 100 + 1) * 100:04d}",
        "raw_unique": int(subset["transcription"].nunique()),
        "lexical_unique": int(norm.nunique()),
        "empty_transcriptions": int(
            subset["transcription"].str.strip().eq("").sum()
        ),
        "duration_sec": float(durations.iloc[0])
    })

qc_table = pd.DataFrame(qc_rows)
assert len(qc_table) == 400
assert (qc_table["lexical_unique"] <= qc_table["raw_unique"]).all()
assert qc_table["empty_transcriptions"].sum() == 0

batch4 = qc_table.loc[
    qc_table["pilot_index"].between(301, 400)
].copy()

batch4_summary = {
    "batch": "0301-0400",
    "status": "PASS",
    "audio_count": 100,
    "hypothesis_count": 500,
    "raw_variable_audio_count": int((batch4["raw_unique"] > 1).sum()),
    "lexically_variable_audio_count": int(
        (batch4["lexical_unique"] > 1).sum()
    ),
    "mean_raw_unique": float(batch4["raw_unique"].mean()),
    "mean_lexical_unique": float(batch4["lexical_unique"].mean()),
    "empty_transcriptions": 0,
    "config_sha256": CONFIG_SHA,
    "asr_csv_sha256": EXPECTED_SHAS["0301-0400"],
    "created_at": datetime.now(timezone.utc).isoformat()
}

assert batch4_summary["raw_variable_audio_count"] == 78
assert np.isclose(batch4_summary["mean_raw_unique"], 2.81)

# Confirm previous QC summaries match current recomputation.
for label in ["0001-0100", "0101-0200", "0201-0300"]:
    old_path = REPORT_DIR / f"step03_batch_{label}_qc.json"
    old_qc = json.loads(old_path.read_text(encoding="utf-8"))
    subset = qc_table.loc[qc_table["batch"] == label]

    assert old_qc["status"] == "PASS"
    assert old_qc["asr_csv_sha256"] == EXPECTED_SHAS[label]
    assert old_qc["raw_variable_audio_count"] == int(
        (subset["raw_unique"] > 1).sum()
    )
    assert old_qc["lexically_variable_audio_count"] == int(
        (subset["lexical_unique"] > 1).sum()
    )

REPORT_DIR.mkdir(parents=True, exist_ok=True)

atomic_csv(batch4, BATCH4_TABLE)
atomic_json(batch4_summary, BATCH4_QC)
atomic_csv(qc_table, FULL_TABLE)

# Preserve original ASR transcript content and generation order.
atomic_csv(combined, FULL_CSV)

full_summary = {
    "status": "PASS",
    "audio_count": 400,
    "hypothesis_count": 2000,
    "batch_count": 4,
    "raw_variable_audio_count": int(
        (qc_table["raw_unique"] > 1).sum()
    ),
    "lexically_variable_audio_count": int(
        (qc_table["lexical_unique"] > 1).sum()
    ),
    "mean_raw_unique": float(qc_table["raw_unique"].mean()),
    "mean_lexical_unique": float(qc_table["lexical_unique"].mean()),
    "empty_transcriptions": 0,
    "config_sha256": CONFIG_SHA,
    "batch_csv_sha256": EXPECTED_SHAS,
    "combined_csv_sha256": sha256(FULL_CSV),
    "created_at": datetime.now(timezone.utc).isoformat()
}

atomic_json(full_summary, FULL_QC)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_full_batch_0301-0400"] = "qc_pass"
progress["step3_full_batch_0301-0400_qc_report"] = str(BATCH4_QC)
progress["step3_full_0400_integrity_status"] = "qc_pass"
progress["step3_full_0400_combined_sha256"] = sha256(FULL_CSV)
progress["step3_full_generated_audio_count"] = 400
progress["step3_full_generated_hypothesis_count"] = 2000
progress["next_task"] = (
    "Create verified final ASR ZIP backup, then proceed to "
    "linguistic feature extraction"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 3.3 BATCH 4 QC SUMMARY ===")
for key, value in batch4_summary.items():
    if key != "created_at":
        print(f"{key}: {value}")

print("\n=== ALL 400 AUDIO INTEGRITY SUMMARY ===")
for key, value in full_summary.items():
    if key not in ["created_at", "batch_csv_sha256"]:
        print(f"{key}: {value}")

print("\n=== LEXICAL VARIABILITY BY BATCH ===")
print(
    qc_table.groupby("batch").agg(
        audio_count=("audio_id", "count"),
        raw_variable=("raw_unique", lambda s: int((s > 1).sum())),
        lexical_variable=("lexical_unique", lambda s: int((s > 1).sum()))
    ).to_string()
)

print("\n=== BATCH 4 LEXICAL DISTRIBUTION ===")
print(
    batch4["lexical_unique"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nCombined CSV:", FULL_CSV)
print("Full QC report:", FULL_QC)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
import zipfile
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
BACKUP = Path("/kaggle/working/asr_fluency_step03_complete.zip")
CHECK = PROJECT / "checkpoints/research_progress.json"

CONFIG_SHA = "1204e1414ec1332ee2f28caf076a799662d28828dadb8e361243140c388cfc77"
FULL_SHA = "b67dfe5cca2329407ab07658809434c2e3bad00d227485f3f8c8c9c1dbbf3e80"

BATCH_SHAS = {
    "0001-0100": "b33b7d0e2564c43738a5f254221e639d53b72edf6f5ced0b407962c152e02fb3",
    "0101-0200": "e838bb9b722c73aeef2cf10dee6e18ea0970e6df29f4c0e5bcf37b142f52595f",
    "0201-0300": "bde72d422c46b9eb357cfc6f7bac702a6a416b302329a1e3582afdae54bfee51",
    "0301-0400": "552a52ee21cbda8a07722f60baafee67114542316d3c602243ef5d0cbae1c777"
}

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

config_file = PROJECT / "configs/asr_hypothesis_pilot_v1.json"
full_file = PROJECT / "features/asr/full/asr_hypotheses_0001-0400.csv"
full_qc_file = PROJECT / "reports/step03_full_0400_integrity_qc.json"

assert sha256(config_file) == CONFIG_SHA
assert sha256(full_file) == FULL_SHA

full_qc = json.loads(full_qc_file.read_text(encoding="utf-8"))

assert full_qc["status"] == "PASS"
assert full_qc["audio_count"] == 400
assert full_qc["hypothesis_count"] == 2000
assert full_qc["config_sha256"] == CONFIG_SHA
assert full_qc["combined_csv_sha256"] == FULL_SHA

# Verify four original batch files and QC reports.
for label, expected_sha in BATCH_SHAS.items():
    csv_file = (
        PROJECT / f"features/asr/full/asr_hypotheses_{label}.csv"
    )
    qc_file = PROJECT / f"reports/step03_batch_{label}_qc.json"

    assert sha256(csv_file) == expected_sha

    qc = json.loads(qc_file.read_text(encoding="utf-8"))
    assert qc["status"] == "PASS"
    assert qc["audio_count"] == 100
    assert qc["hypothesis_count"] == 500
    assert qc["asr_csv_sha256"] == expected_sha

# Verify all 400 individual checkpoints exist.
part_files = []

for label in BATCH_SHAS:
    directory = (
        PROJECT / f"features/asr/full/batch_{label}_parts"
    )
    parts = sorted(directory.glob("asr_*_H1-H5.csv"))

    assert len(parts) == 100, (
        f"Checkpoint count mismatch in {label}: {len(parts)}"
    )
    part_files.extend(parts)

assert len(part_files) == 400

# Check original acoustic results from STEP 2.
acoustic_file = (
    PROJECT / "features/acoustic/pilot_acoustic_features_400.csv"
)

EXPECTED_ACOUSTIC_SHA = (
    "4f477cd0fc0f6cf4382bc92fa39bdae734f22a9440e949b592143db2e261dced"
)

assert acoustic_file.is_file()
assert sha256(acoustic_file) == EXPECTED_ACOUSTIC_SHA

# Validate full ASR CSV once again.
asr = pd.read_csv(
    full_file,
    dtype={"audio_id": "string", "speaker": "string"},
    keep_default_na=False
)

assert len(asr) == 2000
assert asr["audio_id"].nunique() == 400
assert asr["hypothesis_id"].nunique() == 5
assert not asr.duplicated(
    ["audio_id", "hypothesis_id"]
).any()

# Update checkpoint before writing the archive.
progress = json.loads(CHECK.read_text(encoding="utf-8"))

for label in BATCH_SHAS:
    assert progress.get(
        f"step3_full_batch_{label}"
    ) == "qc_pass"

progress["step3_full_0400_integrity_status"] = "qc_pass"
progress["step3_full_generated_audio_count"] = 400
progress["step3_full_generated_hypothesis_count"] = 2000
progress["step3_complete_backup_status"] = "packaging"
progress["step3_complete_backup_filename"] = BACKUP.name
progress["next_task"] = (
    "Verify final STEP 3 backup and start STEP 4 "
    "linguistic feature extraction"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# Archive all small research artifacts, not raw audio or models.
INCLUDE_DIRS = [
    "configs",
    "checkpoints",
    "data/metadata",
    "features/acoustic",
    "features/asr",
    "reports",
    "results"
]

EXCLUDE_SUFFIXES = {
    ".parquet", ".wav", ".mp3", ".flac",
    ".pt", ".pth", ".safetensors", ".tmp",
    ".zip"
}

files = set()

for relative_dir in INCLUDE_DIRS:
    directory = PROJECT / relative_dir

    if not directory.exists():
        continue

    for path in directory.rglob("*"):
        if not path.is_file():
            continue
        if path.suffix.lower() in EXCLUDE_SUFFIXES:
            continue
        files.add(path)

files = sorted(files)

assert config_file in files
assert full_file in files
assert full_qc_file in files
assert acoustic_file in files
assert CHECK in files
assert all(path in files for path in part_files)

manifest = {
    "archive_name": BACKUP.name,
    "research_stage": "STEP 3 complete",
    "status": "QC_PASS",
    "audio_count": 400,
    "hypothesis_count": 2000,
    "individual_asr_checkpoints": 400,
    "frozen_config_sha256": CONFIG_SHA,
    "combined_asr_sha256": FULL_SHA,
    "acoustic_features_sha256": EXPECTED_ACOUSTIC_SHA,
    "raw_audio_included": False,
    "model_weights_included": False,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "files": {}
}

temp_zip = BACKUP.with_suffix(".zip.tmp")

with zipfile.ZipFile(
    temp_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as archive:

    for path in files:
        relative = path.relative_to(PROJECT).as_posix()
        archive_name = (
            "acoustic_linguistic_research/" + relative
        )

        manifest["files"][archive_name] = sha256(path)
        archive.write(path, arcname=archive_name)

    manifest_name = (
        "acoustic_linguistic_research/"
        "backup_manifest_step03_complete.json"
    )

    archive.writestr(
        manifest_name,
        json.dumps(manifest, indent=2, ensure_ascii=False)
    )

# Verify temporary ZIP BEFORE publishing it.
with zipfile.ZipFile(temp_zip, "r") as archive:
    assert archive.testzip() is None

    saved_manifest = json.loads(
        archive.read(manifest_name)
    )
    assert saved_manifest["files"] == manifest["files"]

    for name, expected_hash in manifest["files"].items():
        actual_hash = hashlib.sha256(
            archive.read(name)
        ).hexdigest()
        assert actual_hash == expected_hash, (
            f"Archive integrity failure: {name}"
        )

temp_zip.replace(BACKUP)

# Mark verified status in the live checkpoint.
progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_complete_backup_status"] = "verified"
progress["step3_complete_backup_sha256"] = sha256(BACKUP)
progress["completed_steps"] = sorted(
    set(progress.get("completed_steps", [])) | {3}
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 3 FINAL BACKUP VERIFIED ===")
print("Backup:", BACKUP)
print("Files archived:", len(files))
print("Individual audio checkpoints:", len(part_files))
print("Total audios:", 400)
print("Total hypotheses:", 2000)
print("Full ASR QC:", full_qc["status"])
print("STEP 2 acoustic features: INCLUDED")
print("ZIP size MB:", round(BACKUP.stat().st_size / 1024**2, 3))
print("ZIP SHA256:", sha256(BACKUP))
print("Archive integrity: PASS")
print("STEP 3 status: COMPLETE")
print("Next task:", progress["next_task"])
